# ee.ipynb

`dd.ipynb`의 정상 작동하던 영상 인식/feature extraction 구조를 유지하고, **특징 추출 이후 reasoning 단계**에서 싸움/스토킹만 강화한 전체 통합 파이프라인입니다.

최종 후보는 `fight`, `stalking`, `sidewalk_riding`, `no_helmet`, `electric_kickboard`, `intrusion`, `other`를 모두 유지하며, `final_label`은 영상당 하나만 선택합니다.


## 변경 방향

- `dd.ipynb`에서 잘 되던 YOLO / YOLO-World / SegFormer / feature JSON 생성 코드는 유지합니다.
- 수정 대상은 feature 이후의 판단 함수입니다.
- 싸움과 스토킹은 둘 다 사람 2명 이상 `track-pair box`를 만들고, `sliding window`로 중후반부 발생 상황까지 반영합니다.
- 이륜차 인도 주행, 헬멧 미착용, 전동킥보드, 침입 판단은 그대로 후보에 포함합니다.
- 출력 표에서는 `final_label`이 여러 개 합쳐지지 않고, 가장 confidence가 높은 하나만 표시됩니다.


In [1]:

# ============================================================
# Cell 1. 경로 및 전체 설정  [ee: dd 기반 + fight/stalking 강화]
# ============================================================
from pathlib import Path
import os, json, math, time, gc, warnings, tempfile, shutil, subprocess, sys
from collections import Counter, defaultdict, deque
from statistics import median

CC_VERSION = "ee_full_integrated_dd_base_pair_window_v2"

# ★ 기존 aio3 경로 유지: 여기만 네 환경과 맞으면 나머지는 건드릴 필요 없음
VIDEO_ROOT = Path(r"C:\Users\우성한\Sample (1)\Sample\01.원천데이터")
OUTPUT_ROOT = Path(r"C:\Users\우성한\outputs\prediction")

CC_DIR = OUTPUT_ROOT / "ee"
CC_FEATURE_DIR = CC_DIR / "ee_features"
CC_FINAL_DIR = CC_DIR / "final_ee"
CC_DEBUG_DIR = CC_DIR / "debug"
CC_CROP_DIR = CC_DEBUG_DIR / "crops"
# ee_features가 아직 없을 때 기존 dd_features를 재사용할지 여부.
# 영상 인식/JSON 생성은 dd에서 잘 되던 구조를 유지하므로,
# dd_features가 있으면 다시 YOLO를 돌리지 않고 EE reasoning만 재분석할 수 있다.
REUSE_CC_FEATURES_IF_DD_EMPTY = True
LEGACY_CC_FEATURE_DIR = OUTPUT_ROOT / "dd" / "dd_features"
LEGACY_CC_FEATURE_DIR_2 = OUTPUT_ROOT / "cc" / "cc_features"

for d in [CC_DIR, CC_FEATURE_DIR, CC_FINAL_DIR, CC_DEBUG_DIR, CC_CROP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {
    ".mp4", ".avi", ".mov", ".mkv", ".wmv", ".mpeg", ".mpg", ".m4v",
    ".MP4", ".AVI", ".MOV", ".MKV", ".WMV", ".MPEG", ".MPG", ".M4V"
}

# 전체 처리 시 None. 빠른 테스트는 1~5로 제한.
MAX_VIDEOS = None
MAX_FRAMES_PER_VIDEO = None

# 영상 인식 구조는 dd와 동일하다.
# ee_features를 새로 만들려면 True, 기존 dd_features/ee_features로 reasoning만 다시 돌리려면 False.
FORCE_REBUILD_FEATURES = False

# 샘플링/윈도우
FRAME_INTERVAL = 8          # aio3의 10보다 조금 촘촘하게
WINDOW_SECONDS = 5.0
WINDOW_STRIDE_SECONDS = 1.5

# 현재 노트북 폴더 또는 models 폴더에 업로드/복사한 모델이 있으면 우선 사용
BASE_DIR = Path.cwd()
MODEL_DIR = BASE_DIR / "models"

BASE_YOLO_MODEL_CANDIDATES = [
    BASE_DIR / "yolo11s.pt",
    MODEL_DIR / "yolo11s.pt",
    "yolo11s.pt",
    "yolov8s.pt",
    "yolov8n.pt",
]

POSE_MODEL_CANDIDATES = [
    BASE_DIR / "yolo11s-pose.pt",
    MODEL_DIR / "yolo11s-pose.pt",
    "yolo11s-pose.pt",
    "yolov8s-pose.pt",
    "yolov8n-pose.pt",
]

YOLO_WORLD_MODEL_CANDIDATES = [
    BASE_DIR / "yolov8s-world.pt",
    MODEL_DIR / "yolov8s-world.pt",
    "yolov8s-world.pt",
    "yolov8m-world.pt",
]

# 선택: 직접 학습한 헬멧 모델이 있으면 경로 지정
# 예: HELMET_MODEL_PATH = r"C:\Users\우성한\outputs\prediction\runs_cf6_helmet\bike_helmet_yolov8\weights\best.pt"
HELMET_MODEL_PATH = None

# SegFormer 설정
SEG_MODEL_ID_GPU = "nvidia/segformer-b5-finetuned-cityscapes-1024-1024"
SEG_MODEL_ID_CPU = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
ROAD_ID = 0
SIDEWALK_ID = 1

# COCO 기준 class id
PERSON_CLASS_ID = 0
BICYCLE_CLASS_ID = 1
MOTORCYCLE_CLASS_ID = 3
BASE_TRACK_CLASSES = [PERSON_CLASS_ID, BICYCLE_CLASS_ID, MOTORCYCLE_CLASS_ID]

# YOLO/World confidence
PERSON_MIN_CONF = 0.16
VEHICLE_MIN_CONF = 0.10
WORLD_MIN_CONF = 0.035
POSE_MIN_CONF = 0.12
HELMET_CONF = 0.12
IMGSZ_BASE = 1280
IMGSZ_WORLD = 1280
TRACKER_YAML = "bytetrack.yaml"

# 주행/인도 기준
SIDEWALK_PATCH_RADIUS = 22
SIDEWALK_BAND_Y1 = 0.68
SIDEWALK_BAND_Y2 = 1.00
SIDEWALK_BAND_X_MARGIN = 0.08
SIDEWALK_RIDING_THRESHOLD = 0.36
SIDEWALK_CANDIDATE_THRESHOLD = 0.22

# 헬멧 기준
NO_HELMET_THRESHOLD = 0.34
HELMET_UNKNOWN_MIN_RIDER_FRAMES = 2
SAVE_CROPS = False
MAX_SAVE_CROPS = 200

# 싸움 기준
FIGHT_THRESHOLD = 0.46
FIGHT_CANDIDATE_THRESHOLD = 0.34
FIGHT_CLOSE_DIST = 170
FIGHT_NEAR_DIST = 260
FIGHT_OVERLAP_IOU = 0.015
FIGHT_FLOW_STRONG = 3.0

# 스토킹 기준
STALKING_THRESHOLD = 0.45
STALKING_CANDIDATE_THRESHOLD = 0.34
MIN_TRACK_LEN_FOR_STALKING = 6
FOLLOW_DISTANCE_MIN = 45
FOLLOW_DISTANCE_MAX = 520
FOLLOW_DIRECTION_COS_THRESHOLD = 0.55
FOLLOW_DISTANCE_STABILITY = 135
FOLLOW_BEHIND_PROJECTION_MIN = 15

# 최종 fusion
# 침입은 외부 .py import 없이 현재 노트북 내부 reasoning으로 통합
RUN_INTRUSION_INTEGRATED = True
FINAL_MIN_EVENT_CONF = 0.30

# EE 강화 reasoning 설정
EE_FIGHT_WINDOW_SECONDS = 4.0
EE_FIGHT_WINDOW_STRIDE_SECONDS = 1.0
EE_STALKING_WINDOW_SECONDS = 4.5
EE_STALKING_WINDOW_STRIDE_SECONDS = 1.5
EE_PAIR_MIN_COMMON_FRAMES = 2
EE_FINAL_ONE_LABEL_ONLY = True

# 침입 구조/경계선 crossing 기준
INTRUSION_STRUCTURE_HINTS = {"gate", "entrance", "doorway", "access gate", "fence", "railing", "guard rail", "barrier", "wall", "low wall"}
GATE_STRUCTURE_HINTS = {"gate", "entrance", "doorway", "access gate", "barrier"}
FENCE_STRUCTURE_HINTS = {"fence", "railing", "guard rail", "barrier", "wall", "low wall"}
INTRUSION_THRESHOLD = 0.36
INTRUSION_CANDIDATE_THRESHOLD = 0.25
INTRUSION_MIN_TRACK_LEN = 5
INTRUSION_MIN_MOVEMENT_PX = 35
INTRUSION_MIN_BEFORE_STREAK = 2
INTRUSION_MIN_AFTER_STREAK = 2
INTRUSION_MIN_FRAME_GAP = 2
INTRUSION_LINE_MARGIN = 9
INTRUSION_MAX_STRUCTURE_DISTANCE = 380
INTRUSION_MAX_ID_SWITCH_GAP_SEC = 3.5
INTRUSION_MAX_ID_SWITCH_DIST = 420

# YOLO-World open-vocabulary prompts
WORLD_CLASSES = [
    # people / riders
    "person", "rider", "pedestrian",
    # two-wheelers
    "bicycle", "bike", "electric bike", "motorcycle", "motorbike", "scooter",
    "electric scooter", "kickboard", "moped",
    # helmet/head
    "helmet", "motorcycle helmet", "bike helmet", "no helmet", "bare head",
    "head", "hat", "cap",
    # road scene
    "sidewalk", "walkway", "pavement", "road", "street", "crosswalk",
    "curb", "lane",
    # intrusion structures
    "gate", "entrance gate", "open gate", "metal gate", "access gate", "doorway",
    "fence", "green fence", "metal fence", "railing", "guard rail", "handrail",
    "barrier", "wall", "low wall", "boundary fence",
]

TWO_WHEELER_HINTS = {
    "bicycle", "bike", "electric bike", "motorcycle", "motorbike",
    "scooter", "electric scooter", "kickboard", "moped"
}
SIDEWALK_HINTS = {"sidewalk", "walkway", "pavement", "curb"}
ROAD_HINTS = {"road", "street", "lane", "crosswalk"}
HELMET_POSITIVE_HINTS = {"helmet", "motorcycle helmet", "bike helmet"}
HELMET_NEGATIVE_HINTS = {"no helmet", "bare head", "head"}
HAT_NEUTRAL_HINTS = {"hat", "cap"}

print("CC_VERSION:", CC_VERSION)
print("VIDEO_ROOT exists:", VIDEO_ROOT.exists(), VIDEO_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
if not VIDEO_ROOT.exists():
    raise FileNotFoundError(f"기존 aio3 기준 영상 루트를 찾지 못했습니다: {VIDEO_ROOT}")


CC_VERSION: ee_full_integrated_dd_base_pair_window_v2
VIDEO_ROOT exists: True C:\Users\우성한\Sample (1)\Sample\01.원천데이터
OUTPUT_ROOT: C:\Users\우성한\outputs\prediction


In [2]:

# ============================================================
# Cell 2. 의존성 확인 및 모델 로드
# ============================================================
missing = []

try:
    import cv2
    print("cv2 OK")
except Exception as e:
    missing.append("opencv-python")
    print("cv2 import failed:", repr(e))

try:
    import numpy as np
    print("numpy OK")
except Exception as e:
    missing.append("numpy")
    print("numpy import failed:", repr(e))

try:
    from tqdm import tqdm
    print("tqdm OK")
except Exception as e:
    missing.append("tqdm")
    print("tqdm import failed:", repr(e))

try:
    import torch
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("torch OK, cuda:", torch.cuda.is_available(), "device:", DEVICE)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        print("gpu:", torch.cuda.get_device_name(0))
except Exception as e:
    torch = None
    DEVICE = "cpu"
    missing.append("torch")
    print("torch import failed:", repr(e))

try:
    from ultralytics import YOLO
    print("ultralytics YOLO OK")
except Exception as e:
    YOLO = None
    missing.append("ultralytics")
    print("ultralytics import failed:", repr(e))

try:
    from PIL import Image
    print("PIL OK")
except Exception as e:
    Image = None
    missing.append("pillow")
    print("PIL import failed:", repr(e))

if missing:
    print("설치 필요 후보:", sorted(set(missing)))
    print("예: pip install opencv-python numpy tqdm ultralytics torch transformers pillow")

if YOLO is None:
    raise RuntimeError("ultralytics YOLO를 사용할 수 없습니다.")


def _candidate_to_string(cand):
    if isinstance(cand, Path):
        return str(cand)
    return str(cand)


def load_first_yolo(candidates, label):
    errors = []
    for cand in candidates:
        # Path 후보는 실제 파일이 없으면 skip. 문자열 모델명은 ultralytics cache/download 후보라 시도.
        if isinstance(cand, Path) and not cand.exists():
            errors.append((str(cand), "path_not_exists"))
            continue
        try:
            model = YOLO(_candidate_to_string(cand))
            try:
                model.to(str(DEVICE))
            except Exception:
                pass
            print(f"[model] {label} loaded:", cand)
            return model, str(cand), errors
        except Exception as e:
            errors.append((str(cand), repr(e)))
            print(f"[model] {label} failed:", cand, repr(e))
    return None, None, errors


base_yolo, base_model_name, base_errors = load_first_yolo(BASE_YOLO_MODEL_CANDIDATES, "base_yolo")
if base_yolo is None:
    raise RuntimeError(f"base YOLO 로드 실패: {base_errors[-3:]}")

pose_model, pose_model_name, pose_errors = load_first_yolo(POSE_MODEL_CANDIDATES, "pose_model")
if pose_model is None:
    print("[pose] 사용 불가. 싸움 보조 pose_agitation만 생략합니다.")

world_model = None
world_model_name = None
world_model_active = False
world_errors = []
for cand in YOLO_WORLD_MODEL_CANDIDATES:
    if isinstance(cand, Path) and not cand.exists():
        world_errors.append((str(cand), "path_not_exists"))
        continue
    try:
        wm = YOLO(_candidate_to_string(cand))
        try:
            wm.to(str(DEVICE))
        except Exception:
            pass
        if not hasattr(wm, "set_classes"):
            raise RuntimeError("이 ultralytics 모델은 set_classes를 지원하지 않습니다.")
        wm.set_classes(WORLD_CLASSES)
        world_model = wm
        world_model_name = str(cand)
        world_model_active = True
        print("[world] loaded and set_classes OK:", cand)
        break
    except Exception as e:
        world_errors.append((str(cand), repr(e)))
        print("[world] failed:", cand, repr(e))

if not world_model_active:
    print("[world] 비활성. 기본 YOLO + SegFormer 중심으로 진행합니다.")
    print("[world] errors sample:", world_errors[-3:])

helmet_model = None
helmet_backend = "none"
if HELMET_MODEL_PATH:
    try:
        helmet_model = YOLO(str(HELMET_MODEL_PATH))
        try:
            helmet_model.to(str(DEVICE))
        except Exception:
            pass
        helmet_backend = "local_yolo"
        print("[helmet] local model loaded:", HELMET_MODEL_PATH)
        try:
            print("[helmet] names:", helmet_model.names)
        except Exception:
            pass
    except Exception as e:
        print("[helmet] local model failed:", repr(e))
        helmet_backend = "none"

# SegFormer는 실패해도 bbox fallback으로 계속 진행.
seg_processor = None
seg_model = None
seg_model_id = None
seg_device = None
try:
    from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
    if torch is not None:
        seg_device = DEVICE
        seg_model_id = SEG_MODEL_ID_GPU if str(DEVICE) == "cuda" else SEG_MODEL_ID_CPU
        seg_processor = SegformerImageProcessor.from_pretrained(seg_model_id)
        seg_model = SegformerForSemanticSegmentation.from_pretrained(seg_model_id).to(seg_device)
        seg_model.eval()
        print("[segformer] loaded:", seg_model_id)
except Exception as e:
    print("[segformer] load failed. YOLO-World bbox fallback을 사용합니다:", repr(e))
    seg_processor = None
    seg_model = None

MODEL_STATUS = {
    "cc_version": CC_VERSION,
    "device": str(DEVICE),
    "base_yolo": base_model_name,
    "pose_model": pose_model_name,
    "world_model": world_model_name,
    "world_model_active": world_model_active,
    "world_classes": WORLD_CLASSES if world_model_active else [],
    "segformer": seg_model_id,
    "helmet_backend": helmet_backend,
    "helmet_model": str(HELMET_MODEL_PATH) if HELMET_MODEL_PATH else None,
    "tracker": TRACKER_YAML,
}
print(json.dumps(MODEL_STATUS, ensure_ascii=False, indent=2))


cv2 OK
numpy OK
tqdm OK
torch OK, cuda: True device: cuda
gpu: NVIDIA GeForce RTX 3060
ultralytics YOLO OK
PIL OK
[model] base_yolo loaded: C:\Users\우성한\yolo11s.pt
[model] pose_model loaded: C:\Users\우성한\yolo11s-pose.pt
[world] loaded and set_classes OK: C:\Users\우성한\yolov8s-world.pt


Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

[segformer] loaded: nvidia/segformer-b5-finetuned-cityscapes-1024-1024
{
  "cc_version": "ee_full_integrated_dd_base_pair_window_v2",
  "device": "cuda",
  "base_yolo": "C:\\Users\\우성한\\yolo11s.pt",
  "pose_model": "C:\\Users\\우성한\\yolo11s-pose.pt",
  "world_model": "C:\\Users\\우성한\\yolov8s-world.pt",
  "world_model_active": true,
  "world_classes": [
    "person",
    "rider",
    "pedestrian",
    "bicycle",
    "bike",
    "electric bike",
    "motorcycle",
    "motorbike",
    "scooter",
    "electric scooter",
    "kickboard",
    "moped",
    "helmet",
    "motorcycle helmet",
    "bike helmet",
    "no helmet",
    "bare head",
    "head",
    "hat",
    "cap",
    "sidewalk",
    "walkway",
    "pavement",
    "road",
    "street",
    "crosswalk",
    "curb",
    "lane",
    "gate",
    "entrance gate",
    "open gate",
    "metal gate",
    "access gate",
    "doorway",
    "fence",
    "green fence",
    "metal fence",
    "railing",
    "guard rail",
    "handrail",
    "ba

In [3]:

# ============================================================
# Cell 3. 영상 탐색  [aio3 방식 유지]
# ============================================================
video_paths = sorted([
    p for p in VIDEO_ROOT.rglob("*")
    if p.is_file() and p.suffix in VIDEO_EXTS
])

if MAX_VIDEOS is not None:
    video_paths = video_paths[:MAX_VIDEOS]

print("찾은 영상 수:", len(video_paths))
for p in video_paths[:30]:
    print(p)

if not video_paths:
    raise RuntimeError(
        "기존 VIDEO_ROOT 안에서 영상 파일을 찾지 못했습니다. "
        "이 셀은 경로를 새로 추정하지 않습니다. Cell 1의 VIDEO_ROOT만 확인하세요."
    )


찾은 영상 수: 22
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\전동킥보드 인도 주행\es_e3768\es_e3768_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데

In [4]:

# ============================================================
# Cell 4. 공통 유틸: bbox, 모델 출력 정규화, segmentation, rider/head-zone
# ============================================================
def safe_stem_from_path(path: Path, root: Path):
    rel = path.relative_to(root)
    return "__".join(rel.with_suffix("").parts).replace(" ", "_")


def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def bbox_area(b):
    x1, y1, x2, y2 = b
    return max(0.0, float(x2) - float(x1)) * max(0.0, float(y2) - float(y1))


def bbox_center(b):
    x1, y1, x2, y2 = b
    return [round((float(x1) + float(x2)) / 2, 2), round((float(y1) + float(y2)) / 2, 2)]


def bbox_bottom_center(b):
    x1, y1, x2, y2 = b
    return [round((float(x1) + float(x2)) / 2, 2), round(float(y2), 2)]


def bbox_head_zone(b, frame_w=None, frame_h=None, expand=0.18):
    x1, y1, x2, y2 = [float(v) for v in b]
    w = max(1.0, x2 - x1)
    h = max(1.0, y2 - y1)
    hx1 = x1 - w * expand
    hx2 = x2 + w * expand
    hy1 = y1 - h * 0.08
    hy2 = y1 + h * 0.34
    if frame_w is not None:
        hx1 = clamp(hx1, 0, frame_w - 1)
        hx2 = clamp(hx2, 0, frame_w - 1)
    if frame_h is not None:
        hy1 = clamp(hy1, 0, frame_h - 1)
        hy2 = clamp(hy2, 0, frame_h - 1)
    return [int(round(hx1)), int(round(hy1)), int(round(hx2)), int(round(hy2))]


def bbox_expand(b, frame_w, frame_h, ratio=0.10):
    x1, y1, x2, y2 = [float(v) for v in b]
    w = x2 - x1
    h = y2 - y1
    return [
        int(clamp(round(x1 - w * ratio), 0, frame_w - 1)),
        int(clamp(round(y1 - h * ratio), 0, frame_h - 1)),
        int(clamp(round(x2 + w * ratio), 0, frame_w - 1)),
        int(clamp(round(y2 + h * ratio), 0, frame_h - 1)),
    ]


def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = [float(v) for v in a]
    bx1, by1, bx2, by2 = [float(v) for v in b]
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = bbox_area(a) + bbox_area(b) - inter
    return inter / union if union > 0 else 0.0


def bbox_overlap_ratio(a, b):
    # a 면적 기준으로 b와 겹치는 비율
    ax1, ay1, ax2, ay2 = [float(v) for v in a]
    bx1, by1, bx2, by2 = [float(v) for v in b]
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area = bbox_area(a)
    return inter / area if area > 0 else 0.0


def point_in_bbox(pt, b, margin=0):
    x, y = float(pt[0]), float(pt[1])
    x1, y1, x2, y2 = [float(v) for v in b]
    return (x1 - margin) <= x <= (x2 + margin) and (y1 - margin) <= y <= (y2 + margin)


def point_to_bbox_distance(pt, b):
    x, y = float(pt[0]), float(pt[1])
    x1, y1, x2, y2 = [float(v) for v in b]
    dx = max(x1 - x, 0, x - x2)
    dy = max(y1 - y, 0, y - y2)
    return math.sqrt(dx * dx + dy * dy)


def dist(a, b):
    return math.sqrt((float(a[0]) - float(b[0])) ** 2 + (float(a[1]) - float(b[1])) ** 2)


def vec(a, b):
    return [float(b[0]) - float(a[0]), float(b[1]) - float(a[1])]


def norm(v):
    return math.sqrt(float(v[0]) ** 2 + float(v[1]) ** 2)


def cosine(a, b):
    na, nb = norm(a), norm(b)
    if na < 1e-6 or nb < 1e-6:
        return 0.0
    return (float(a[0]) * float(b[0]) + float(a[1]) * float(b[1])) / (na * nb)


def video_meta(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return {
        "fps": round(float(fps), 3),
        "width": w,
        "height": h,
        "frame_count": n,
        "length_sec": round(float(n / fps), 3) if fps else 0,
    }


def normalize_label(label):
    return str(label).strip().lower().replace("_", " ")


def is_two_wheeler_label(label):
    label = normalize_label(label)
    return label in TWO_WHEELER_HINTS or any(k in label for k in ["motorcycle", "motorbike", "scooter", "kickboard", "bicycle", "bike"])


def is_sidewalk_label(label):
    label = normalize_label(label)
    return label in SIDEWALK_HINTS or any(k in label for k in ["sidewalk", "walkway", "pavement", "curb"])


def is_road_label(label):
    label = normalize_label(label)
    return label in ROAD_HINTS or any(k in label for k in ["road", "street", "lane", "crosswalk"])


def is_helmet_positive_label(label):
    label = normalize_label(label)
    return label in HELMET_POSITIVE_HINTS or ("helmet" in label and "no" not in label and "without" not in label)


def is_helmet_negative_label(label):
    label = normalize_label(label)
    return label in HELMET_NEGATIVE_HINTS or "no helmet" in label or "without helmet" in label or "bare head" in label


def is_head_or_hat_label(label):
    label = normalize_label(label)
    return label in HAT_NEUTRAL_HINTS or label == "head" or "cap" in label or "hat" in label


def yolo_result_to_objects(result, source_name, min_conf=0.0):
    objects = []
    if result is None or result.boxes is None:
        return objects

    boxes = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy()
    clss = result.boxes.cls.cpu().numpy()
    ids = None
    if getattr(result.boxes, "id", None) is not None:
        try:
            ids = result.boxes.id.cpu().numpy()
        except Exception:
            ids = None

    names = getattr(result, "names", None)
    if names is None:
        names = getattr(result, "names", {})

    for idx, (box, conf, cls_id) in enumerate(zip(boxes, confs, clss)):
        conf = float(conf)
        if conf < min_conf:
            continue
        cls_id = int(cls_id)
        if isinstance(names, dict):
            label = names.get(cls_id, str(cls_id))
        else:
            try:
                label = names[cls_id]
            except Exception:
                label = str(cls_id)

        x1, y1, x2, y2 = box.tolist()
        bbox = [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]
        if bbox_area(bbox) < 20:
            continue

        obj = {
            "source": source_name,
            "class_id": cls_id,
            "class": normalize_label(label),
            "confidence": round(conf, 4),
            "bbox": bbox,
            "center": bbox_center(bbox),
            "bottom_center": bbox_bottom_center(bbox),
            "area": round(float(bbox_area(bbox)), 2),
        }
        if ids is not None and idx < len(ids):
            obj["track_id"] = int(ids[idx])
        objects.append(obj)
    return objects


def reset_yolo_tracker(model):
    try:
        model.predictor = None
    except Exception:
        pass


def run_base_yolo_frame(frame):
    # 사람/자전거/오토바이를 한 번에 track. 실패하면 predict로 fallback.
    try:
        results = base_yolo.track(
            source=frame,
            persist=True,
            tracker=TRACKER_YAML,
            classes=BASE_TRACK_CLASSES,
            conf=min(PERSON_MIN_CONF, VEHICLE_MIN_CONF),
            imgsz=IMGSZ_BASE,
            device=0 if str(DEVICE) == "cuda" else "cpu",
            verbose=False
        )
        return yolo_result_to_objects(results[0], "base_track", min_conf=min(PERSON_MIN_CONF, VEHICLE_MIN_CONF)), "track"
    except Exception as e:
        results = base_yolo.predict(
            source=frame,
            classes=BASE_TRACK_CLASSES,
            conf=min(PERSON_MIN_CONF, VEHICLE_MIN_CONF),
            imgsz=IMGSZ_BASE,
            device=0 if str(DEVICE) == "cuda" else "cpu",
            verbose=False
        )
        obs = yolo_result_to_objects(results[0], "base_predict", min_conf=min(PERSON_MIN_CONF, VEHICLE_MIN_CONF))
        return obs, "predict_fallback"


def run_world_frame(frame):
    if not world_model_active or world_model is None:
        return []
    try:
        results = world_model.predict(
            source=frame,
            conf=WORLD_MIN_CONF,
            imgsz=IMGSZ_WORLD,
            device=0 if str(DEVICE) == "cuda" else "cpu",
            verbose=False
        )
        return yolo_result_to_objects(results[0], "yolo_world", min_conf=WORLD_MIN_CONF)
    except Exception as e:
        return [{"source": "yolo_world_error", "error": repr(e)}]


def merge_duplicate_objects(objects, iou_thr=0.55):
    # 같은 종류끼리 bbox가 크게 겹치면 하나로 합침. track_id가 있거나 confidence가 높은 것을 우선.
    kept = []
    for obj in sorted(objects, key=lambda x: (1 if x.get("track_id") is not None else 0, x.get("confidence", 0)), reverse=True):
        merged = False
        for k in kept:
            same_kind = (
                obj.get("class") == k.get("class")
                or (is_two_wheeler_label(obj.get("class")) and is_two_wheeler_label(k.get("class")))
            )
            if same_kind and bbox_iou(obj["bbox"], k["bbox"]) >= iou_thr:
                k.setdefault("merged_sources", [k.get("source")])
                k["merged_sources"].append(obj.get("source"))
                k["confidence"] = max(k.get("confidence", 0), obj.get("confidence", 0))
                if k.get("track_id") is None and obj.get("track_id") is not None:
                    k["track_id"] = obj.get("track_id")
                merged = True
                break
        if not merged:
            item = dict(obj)
            item["merged_sources"] = [obj.get("source")]
            kept.append(item)
    return kept


def optical_flow_stats(prev_gray, gray):
    if prev_gray is None or gray is None:
        return {"mean_mag": 0.0, "max_mag": 0.0, "strong_ratio": 0.0}
    h, w = gray.shape[:2]
    if w > 480:
        scale = 480 / w
        gray_s = cv2.resize(gray, (480, int(h * scale)))
        prev_s = cv2.resize(prev_gray, (480, int(h * scale)))
    else:
        gray_s = gray
        prev_s = prev_gray
    flow = cv2.calcOpticalFlowFarneback(prev_s, gray_s, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    return {
        "mean_mag": round(float(np.mean(mag)), 4),
        "max_mag": round(float(np.max(mag)), 4),
        "strong_ratio": round(float(np.mean(mag >= FIGHT_FLOW_STRONG)), 4),
    }


def get_seg_mask(frame):
    if seg_processor is None or seg_model is None or Image is None:
        return None
    try:
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        inputs = seg_processor(images=image, return_tensors="pt")
        if torch is not None:
            inputs = {k: v.to(seg_device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = seg_model(**inputs)
        logits = outputs.logits
        up = torch.nn.functional.interpolate(
            logits,
            size=frame.shape[:2],
            mode="bilinear",
            align_corners=False
        )
        pred = up.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
        return pred
    except Exception as e:
        return None


def patch_label_ratio(seg_mask, x, y, radius=SIDEWALK_PATCH_RADIUS):
    if seg_mask is None:
        return {"seg_sidewalk_ratio": None, "seg_road_ratio": None}
    h, w = seg_mask.shape[:2]
    x = int(round(x))
    y = int(round(y))
    x1, x2 = max(0, x - radius), min(w, x + radius + 1)
    y1, y2 = max(0, y - radius), min(h, y + radius + 1)
    patch = seg_mask[y1:y2, x1:x2]
    if patch.size == 0:
        return {"seg_sidewalk_ratio": 0.0, "seg_road_ratio": 0.0}
    return {
        "seg_sidewalk_ratio": round(float(np.mean(patch == SIDEWALK_ID)), 4),
        "seg_road_ratio": round(float(np.mean(patch == ROAD_ID)), 4),
    }


def bbox_bottom_band_label_ratio(seg_mask, bbox):
    if seg_mask is None:
        return {"band_sidewalk_ratio": None, "band_road_ratio": None}
    h, w = seg_mask.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    bw = max(1, x2 - x1)
    bh = max(1, y2 - y1)
    bx1 = int(clamp(x1 + bw * SIDEWALK_BAND_X_MARGIN, 0, w - 1))
    bx2 = int(clamp(x2 - bw * SIDEWALK_BAND_X_MARGIN, 0, w - 1))
    by1 = int(clamp(y1 + bh * SIDEWALK_BAND_Y1, 0, h - 1))
    by2 = int(clamp(y1 + bh * SIDEWALK_BAND_Y2, 0, h - 1))
    patch = seg_mask[by1:by2 + 1, bx1:bx2 + 1]
    if patch.size == 0:
        return {"band_sidewalk_ratio": 0.0, "band_road_ratio": 0.0}
    return {
        "band_sidewalk_ratio": round(float(np.mean(patch == SIDEWALK_ID)), 4),
        "band_road_ratio": round(float(np.mean(patch == ROAD_ID)), 4),
    }


def scene_bbox_support_for_point(point, tw_bbox, scene_objects):
    sidewalk_scores = []
    road_scores = []
    best_sidewalk = None
    best_road = None

    for obj in scene_objects:
        label = obj.get("class", obj.get("label", ""))
        if "bbox" not in obj:
            continue
        ob = obj["bbox"]
        # point가 들어가거나 이륜차 bbox와 어느 정도 겹치면 근거로 사용
        p_inside = point_in_bbox(point, ob, margin=8)
        overlap_tw = bbox_overlap_ratio(tw_bbox, ob)
        score = 0.0
        if p_inside:
            score += 0.70
        score += min(0.30, overlap_tw * 1.4)
        score *= max(0.25, float(obj.get("confidence", 0.0)))

        if is_sidewalk_label(label):
            sidewalk_scores.append(score)
            if best_sidewalk is None or score > best_sidewalk.get("_support", 0):
                best_sidewalk = {**obj, "_support": round(float(score), 4)}
        elif is_road_label(label):
            road_scores.append(score)
            if best_road is None or score > best_road.get("_support", 0):
                best_road = {**obj, "_support": round(float(score), 4)}

    return {
        "world_sidewalk_bbox_score": round(float(max(sidewalk_scores) if sidewalk_scores else 0.0), 4),
        "world_road_bbox_score": round(float(max(road_scores) if road_scores else 0.0), 4),
        "best_sidewalk_obj": best_sidewalk,
        "best_road_obj": best_road,
    }


def compute_sidewalk_status_for_vehicle(tw_obj, seg_mask, scene_objects):
    bbox = tw_obj["bbox"]
    point = bbox_bottom_center(bbox)
    ratios = patch_label_ratio(seg_mask, point[0], point[1], SIDEWALK_PATCH_RADIUS)
    band = bbox_bottom_band_label_ratio(seg_mask, bbox)
    bbox_support = scene_bbox_support_for_point(point, bbox, scene_objects)

    seg_sw = ratios.get("seg_sidewalk_ratio")
    seg_road = ratios.get("seg_road_ratio")
    band_sw = band.get("band_sidewalk_ratio")
    band_road = band.get("band_road_ratio")

    scores = []
    if seg_sw is not None:
        scores.append(float(seg_sw) * 0.55)
    if band_sw is not None:
        scores.append(float(band_sw) * 0.75)
    if bbox_support["world_sidewalk_bbox_score"] > 0:
        scores.append(float(bbox_support["world_sidewalk_bbox_score"]) * 0.90)

    road_scores = []
    if seg_road is not None:
        road_scores.append(float(seg_road) * 0.55)
    if band_road is not None:
        road_scores.append(float(band_road) * 0.75)
    if bbox_support["world_road_bbox_score"] > 0:
        road_scores.append(float(bbox_support["world_road_bbox_score"]) * 0.90)

    sidewalk_score = max(scores) if scores else 0.0
    road_score = max(road_scores) if road_scores else 0.0

    if sidewalk_score >= SIDEWALK_RIDING_THRESHOLD and sidewalk_score >= road_score * 0.85:
        status = "sidewalk"
    elif sidewalk_score >= SIDEWALK_CANDIDATE_THRESHOLD:
        status = "sidewalk_candidate"
    elif road_score >= 0.22 and road_score > sidewalk_score:
        status = "road"
    else:
        status = "unknown"

    return {
        "two_wheeler_class": tw_obj.get("class"),
        "two_wheeler_source": tw_obj.get("source"),
        "two_wheeler_track_id": tw_obj.get("track_id"),
        "two_wheeler_confidence": tw_obj.get("confidence"),
        "bbox": bbox,
        "bottom_center": point,
        "sidewalk_score": round(float(sidewalk_score), 4),
        "road_score": round(float(road_score), 4),
        "sidewalk_status": status,
        **ratios,
        **band,
        **bbox_support,
    }


def rider_pair_score(person, tw):
    pb = person["bbox"]
    tb = tw["bbox"]
    pc = person.get("center", bbox_center(pb))
    tc = tw.get("center", bbox_center(tb))
    pfoot = bbox_bottom_center(pb)
    tcenter = bbox_center(tb)
    d_center = dist(pc, tc)
    d_foot = dist(pfoot, tcenter)
    iou = bbox_iou(pb, tb)
    p_overlap = bbox_overlap_ratio(pb, tb)
    tw_overlap = bbox_overlap_ratio(tb, pb)

    # 사람 발점이 이륜차 bbox와 가까우면 rider 가능성이 큼
    foot_to_tw = point_to_bbox_distance(pfoot, tb)

    ph = max(1.0, pb[3] - pb[1])
    tw_diag = math.sqrt(max(1.0, bbox_area(tb)))
    norm_d = min(1.0, d_center / max(160.0, ph * 1.2, tw_diag * 1.2))
    norm_foot = min(1.0, foot_to_tw / max(80.0, tw_diag * 0.5))

    score = 0.0
    score += (1.0 - norm_d) * 0.28
    score += (1.0 - norm_foot) * 0.34
    score += min(0.18, iou * 5.0)
    score += min(0.12, p_overlap * 1.8)
    score += min(0.08, tw_overlap * 1.5)

    # 사람 bbox 하단이 이륜차와 비슷한 높이에 있으면 보너스
    vertical_ok = abs(float(pb[3]) - float(tb[3])) <= max(80.0, ph * 0.45)
    if vertical_ok:
        score += 0.08

    return round(float(min(1.0, score)), 4), {
        "center_distance": round(float(d_center), 2),
        "foot_to_vehicle_distance": round(float(foot_to_tw), 2),
        "foot_to_vehicle_center_distance": round(float(d_foot), 2),
        "iou": round(float(iou), 4),
        "person_overlap_ratio": round(float(p_overlap), 4),
        "vehicle_overlap_ratio": round(float(tw_overlap), 4),
        "vertical_ok": bool(vertical_ok),
    }


def build_rider_pairs(persons, two_wheelers, min_score=0.22):
    pairs = []
    for pi, p in enumerate(persons):
        for ti, tw in enumerate(two_wheelers):
            score, detail = rider_pair_score(p, tw)
            if score < min_score:
                continue
            pairs.append({
                "person_index": pi,
                "two_wheeler_index": ti,
                "person_track_id": p.get("track_id"),
                "two_wheeler_track_id": tw.get("track_id"),
                "person_bbox": p["bbox"],
                "two_wheeler_bbox": tw["bbox"],
                "two_wheeler_class": tw.get("class"),
                "score": score,
                **detail,
            })
    # 같은 person/vehicle에 대해 너무 많은 pair가 생기지 않게 상위만 유지
    pairs = sorted(pairs, key=lambda x: x["score"], reverse=True)
    used_person = set()
    used_tw = set()
    selected = []
    for pair in pairs:
        key_p = pair["person_index"]
        key_tw = pair["two_wheeler_index"]
        if key_p in used_person and key_tw in used_tw:
            continue
        selected.append(pair)
        used_person.add(key_p)
        used_tw.add(key_tw)
    return selected[:8]


def extract_pose_people(result):
    people = []
    if result is None or result.boxes is None:
        return people
    boxes = result.boxes.xyxy.cpu().numpy() if result.boxes is not None else []
    confs = result.boxes.conf.cpu().numpy() if result.boxes is not None else []
    keypoints_xy = None
    keypoints_conf = None
    if result.keypoints is not None and result.keypoints.xy is not None:
        keypoints_xy = result.keypoints.xy.cpu().numpy()
        if result.keypoints.conf is not None:
            keypoints_conf = result.keypoints.conf.cpu().numpy()

    for idx, box in enumerate(boxes):
        x1, y1, x2, y2 = box.tolist()
        item = {
            "bbox": [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))],
            "confidence": round(float(confs[idx]), 4) if idx < len(confs) else None,
            "keypoints": None,
        }
        if keypoints_xy is not None and idx < len(keypoints_xy):
            kpts = []
            for k, xy in enumerate(keypoints_xy[idx]):
                kp = {"index": k, "x": round(float(xy[0]), 2), "y": round(float(xy[1]), 2)}
                if keypoints_conf is not None:
                    kp["confidence"] = round(float(keypoints_conf[idx][k]), 4)
                kpts.append(kp)
            item["keypoints"] = kpts
        people.append(item)
    return people


def compute_pose_agitation(pose_people):
    vals = []
    # COCO: wrist 9/10, elbow 7/8, shoulder 5/6
    for p in pose_people:
        kpts = p.get("keypoints") or []
        valid = [kp for kp in kpts if kp.get("confidence", 1.0) >= 0.18]
        if len(valid) < 5:
            continue
        by = {kp["index"]: kp for kp in valid}
        for a, b in [(5, 7), (7, 9), (6, 8), (8, 10)]:
            if a in by and b in by:
                vals.append(dist([by[a]["x"], by[a]["y"]], [by[b]["x"], by[b]["y"]]))
    if not vals:
        return 0.0
    return round(float(min(1.0, np.std(vals) / 80.0)), 4)


def run_pose_frame(frame):
    if pose_model is None:
        return [], 0.0
    try:
        pr = pose_model.predict(
            source=frame,
            conf=POSE_MIN_CONF,
            imgsz=IMGSZ_BASE,
            device=0 if str(DEVICE) == "cuda" else "cpu",
            verbose=False
        )[0]
        people = extract_pose_people(pr)
        return people, compute_pose_agitation(people)
    except Exception:
        return [], 0.0


def predict_helmet_local(frame, head_box):
    if helmet_model is None:
        return []
    x1, y1, x2, y2 = [int(v) for v in head_box]
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return []
    try:
        results = helmet_model.predict(
            source=crop,
            conf=HELMET_CONF,
            imgsz=640,
            device=0 if str(DEVICE) == "cuda" else "cpu",
            verbose=False
        )
        dets = yolo_result_to_objects(results[0], "helmet_local", min_conf=HELMET_CONF)
        # crop 좌표를 frame 좌표로 복원
        for d in dets:
            bx = d["bbox"]
            d["bbox_in_frame"] = [bx[0] + x1, bx[1] + y1, bx[2] + x1, bx[3] + y1]
        return dets
    except Exception as e:
        return [{"source": "helmet_local_error", "error": repr(e)}]


def judge_helmet_for_pair(frame, pair, world_objects, frame_w, frame_h, frame_id, video_stem):
    head_box = bbox_head_zone(pair["person_bbox"], frame_w=frame_w, frame_h=frame_h)
    local_dets = predict_helmet_local(frame, head_box) if helmet_model is not None else []

    pos_score = 0.0
    neg_score = 0.0
    neutral_head_score = 0.0
    matched_world = []

    # local helmet model 결과
    for d in local_dets:
        label = d.get("class", "")
        conf = float(d.get("confidence", 0.0))
        if is_helmet_positive_label(label):
            pos_score = max(pos_score, conf)
        elif is_helmet_negative_label(label):
            neg_score = max(neg_score, conf)

    # YOLO-World 결과를 head-zone과 매칭
    for obj in world_objects:
        label = obj.get("class", "")
        if not (is_helmet_positive_label(label) or is_helmet_negative_label(label) or is_head_or_hat_label(label)):
            continue
        ob = obj.get("bbox")
        if not ob:
            continue
        iou_h = bbox_iou(head_box, ob)
        overlap_h = bbox_overlap_ratio(ob, head_box)
        center_inside = point_in_bbox(obj.get("center", bbox_center(ob)), head_box, margin=10)
        match_score = max(iou_h, overlap_h, 0.55 if center_inside else 0.0) * float(obj.get("confidence", 0.0))
        if match_score <= 0:
            continue
        item = {
            "label": label,
            "confidence": obj.get("confidence"),
            "bbox": ob,
            "match_score": round(float(match_score), 4),
            "iou_head": round(float(iou_h), 4),
            "overlap_head": round(float(overlap_h), 4),
            "center_inside_head": bool(center_inside),
        }
        matched_world.append(item)
        if is_helmet_positive_label(label):
            pos_score = max(pos_score, match_score)
        elif is_helmet_negative_label(label):
            neg_score = max(neg_score, match_score)
        else:
            neutral_head_score = max(neutral_head_score, match_score)

    # 판정 규칙
    if pos_score >= max(0.10, neg_score * 1.15):
        status = "helmet"
        reason = "head-zone 안에서 helmet 후보가 가장 강함"
        confidence = min(0.95, 0.45 + pos_score)
    elif neg_score >= 0.08 and neg_score >= pos_score * 0.95:
        status = "no_helmet"
        reason = "head-zone 안에서 no helmet/bare head/head 후보가 helmet보다 강함"
        confidence = min(0.90, 0.42 + neg_score)
    elif world_model_active and neutral_head_score >= 0.08 and pos_score < 0.06:
        status = "no_helmet_candidate"
        reason = "head는 잡혔지만 helmet 후보가 약함. 후보로만 사용"
        confidence = min(0.68, 0.30 + neutral_head_score)
    else:
        status = "unknown"
        if helmet_model is None and not world_model_active:
            reason = "helmet 전용 모델과 YOLO-World가 모두 없어 판단 불가"
        elif not matched_world and not local_dets:
            reason = "rider는 있으나 head-zone 내부 helmet/head 후보가 없음"
        else:
            reason = "helmet/no-helmet 근거가 서로 약하거나 불충분"
        confidence = 0.0

    return {
        "frame_id": frame_id,
        "person_track_id": pair.get("person_track_id"),
        "two_wheeler_track_id": pair.get("two_wheeler_track_id"),
        "head_box": head_box,
        "helmet_status": status,
        "helmet_confidence": round(float(confidence), 4),
        "helmet_score": round(float(pos_score), 4),
        "no_helmet_score": round(float(neg_score), 4),
        "neutral_head_score": round(float(neutral_head_score), 4),
        "reason": reason,
        "world_matches": matched_world[:8],
        "local_detections": local_dets[:8],
    }


In [5]:

# ============================================================
# Cell 5. EE 프레임 특징 JSON 생성  [dd 영상 인식 구조 유지]
#   - base YOLO: person/bicycle/motorcycle tracking
#   - YOLO-World: 이륜차/인도/헬멧/침입 구조 보조 bbox
#   - SegFormer: 인도/도로 픽셀 근거
# ============================================================
def create_cc_features_for_video(video_path: Path, output_json_path: Path):
    meta = video_meta(video_path)
    fps = meta.get("fps") or 30.0
    frame_w = int(meta.get("width", 0))
    frame_h = int(meta.get("height", 0))

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")

    reset_yolo_tracker(base_yolo)

    frames = []
    frame_idx = 0
    sampled_idx = 0
    prev_gray = None
    mode_counter = Counter()
    video_stem = safe_stem_from_path(video_path, VIDEO_ROOT)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if MAX_FRAMES_PER_VIDEO is not None and frame_idx >= MAX_FRAMES_PER_VIDEO:
            break

        if frame_idx % FRAME_INTERVAL == 0:
            time_sec = frame_idx / fps if fps else sampled_idx

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            flow = optical_flow_stats(prev_gray, gray)
            prev_gray = gray

            base_objects, tracker_mode = run_base_yolo_frame(frame)
            mode_counter[tracker_mode] += 1

            # confidence/class 기준 분리
            persons = []
            base_tw = []
            for obj in base_objects:
                cls_id = obj.get("class_id")
                label = obj.get("class")
                if cls_id == PERSON_CLASS_ID or label == "person":
                    if obj.get("confidence", 0) >= PERSON_MIN_CONF:
                        item = dict(obj)
                        item["obj_label"] = "person"
                        persons.append(item)
                elif cls_id in [BICYCLE_CLASS_ID, MOTORCYCLE_CLASS_ID] or is_two_wheeler_label(label):
                    if obj.get("confidence", 0) >= VEHICLE_MIN_CONF:
                        item = dict(obj)
                        item["obj_label"] = "two_wheeler"
                        base_tw.append(item)

            world_objects_raw = run_world_frame(frame)
            world_errors = [o for o in world_objects_raw if o.get("source") == "yolo_world_error"]
            world_objects = [o for o in world_objects_raw if "bbox" in o]

            world_tw = []
            scene_surface_objects = []
            intrusion_structure_objects = []
            world_helmet_head_objects = []

            for obj in world_objects:
                label = obj.get("class", "")
                if is_two_wheeler_label(label):
                    item = dict(obj)
                    item["obj_label"] = "two_wheeler"
                    world_tw.append(item)
                if is_sidewalk_label(label) or is_road_label(label):
                    scene_surface_objects.append(obj)
                if any(k in label for k in ["gate", "fence", "railing", "barrier", "wall"]):
                    intrusion_structure_objects.append(obj)
                if is_helmet_positive_label(label) or is_helmet_negative_label(label) or is_head_or_hat_label(label):
                    world_helmet_head_objects.append(obj)

            two_wheelers = merge_duplicate_objects(base_tw + world_tw, iou_thr=0.50)
            rider_pairs = build_rider_pairs(persons, two_wheelers, min_score=0.22)

            # segmentation + world bbox로 인도 여부 추출
            seg_mask = get_seg_mask(frame)
            sidewalk_evidence = []
            for tw_idx, tw in enumerate(two_wheelers):
                ev = compute_sidewalk_status_for_vehicle(tw, seg_mask, scene_surface_objects)
                ev["two_wheeler_index"] = tw_idx
                # rider pair와 연결 여부
                related_pairs = [p for p in rider_pairs if p.get("two_wheeler_index") == tw_idx]
                ev["has_rider_pair"] = bool(related_pairs)
                ev["best_rider_pair_score"] = max([p.get("score", 0) for p in related_pairs], default=0.0)
                sidewalk_evidence.append(ev)

            # pose는 매 프레임 돌리면 느릴 수 있으나, 싸움 영상은 보조 근거가 중요해서 샘플 프레임마다 수행
            pose_people, pose_agitation = run_pose_frame(frame)

            # 헬멧 evidence: rider pair별 head-zone만 검사
            helmet_evidence = []
            for pair in rider_pairs:
                helmet_evidence.append(
                    judge_helmet_for_pair(
                        frame=frame,
                        pair=pair,
                        world_objects=world_helmet_head_objects,
                        frame_w=frame_w,
                        frame_h=frame_h,
                        frame_id=frame_idx,
                        video_stem=video_stem,
                    )
                )

            # frame 저장
            frames.append({
                "frame_id": frame_idx,
                "sampled_id": sampled_idx,
                "time_sec": round(float(time_sec), 3),
                "tracker_mode": tracker_mode,
                "base_objects": base_objects,
                "world_objects": world_objects,
                "world_errors": world_errors,
                "persons": persons,
                "two_wheelers": two_wheelers,
                "rider_pairs": rider_pairs,
                "sidewalk_evidence": sidewalk_evidence,
                "helmet_evidence": helmet_evidence,
                "world_helmet_head_objects": world_helmet_head_objects,
                "scene_surface_objects": scene_surface_objects,
                "intrusion_structure_objects": intrusion_structure_objects,
                "segformer_used": seg_mask is not None,
                "pose_people": pose_people,
                "pose_agitation": round(float(pose_agitation), 4),
                "flow": flow,
            })
            sampled_idx += 1

        frame_idx += 1

    cap.release()

    data = {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "relative_video_path": str(video_path.relative_to(VIDEO_ROOT)),
        "meta": meta,
        "settings": {
            "cc_version": CC_VERSION,
            "frame_interval": FRAME_INTERVAL,
            "window_seconds": WINDOW_SECONDS,
            "window_stride_seconds": WINDOW_STRIDE_SECONDS,
            "model_status": MODEL_STATUS,
            "tracker_modes": dict(mode_counter),
        },
        "frames": frames,
    }

    output_json_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return data


created = 0
used = 0
failed = []

for video_path in tqdm(video_paths, desc="EE feature JSON"):
    stem = safe_stem_from_path(video_path, VIDEO_ROOT)
    out_path = CC_FEATURE_DIR / f"{stem}.json"

    if out_path.exists() and not FORCE_REBUILD_FEATURES:
        used += 1
        continue

    try:
        create_cc_features_for_video(video_path, out_path)
        created += 1
    except Exception as e:
        failed.append({"video": str(video_path), "error": repr(e)})
        print("[feature failed]", video_path, repr(e))

print("=" * 80)
print("EE feature 생성 완료")
print("전체 영상:", len(video_paths))
print("생성:", created)
print("기존 사용:", used)
print("실패:", len(failed))
if failed:
    for item in failed[:20]:
        print(item)


EE feature JSON: 100%|█████████████████████████████████████████████████████████████████| 22/22 [19:11<00:00, 52.34s/it]

EE feature 생성 완료
전체 영상: 22
생성: 18
기존 사용: 4
실패: 0


In [6]:

# ============================================================
# Cell 6. Reasoning 함수 + DD 통합 침입  [EE에서는 아래 Cell 6-B가 일부 함수 override]
#   - 사람/이륜차 track 재구성
#   - 인도 주행 / 헬멧 / 싸움 / 스토킹 점수화
# ============================================================
def assign_tracks_from_frame_objects(frames, object_key="persons", fallback_max_dist=160, min_conf=0.0):
    # 1순위: YOLO ByteTrack track_id 사용
    tracked = defaultdict(list)
    tracked_obs_count = 0
    total_count = 0

    for fr in frames:
        for obj in fr.get(object_key, []):
            if obj.get("confidence", 0) < min_conf:
                continue
            total_count += 1
            tid = obj.get("track_id")
            if tid is None:
                continue
            item = dict(obj)
            item["track_id"] = int(tid)
            item["frame_id"] = fr.get("frame_id")
            item["time_sec"] = fr.get("time_sec")
            item["track_source"] = "yolo_bytetrack"
            tracked[int(tid)].append(item)
            tracked_obs_count += 1

    if total_count > 0 and tracked_obs_count / max(1, total_count) >= 0.35:
        return dict(tracked)

    # fallback: center 기반 greedy tracker
    next_id = 1
    active = {}
    history = defaultdict(list)

    for fr in frames:
        objs = sorted(fr.get(object_key, []), key=lambda x: -x.get("confidence", 0))
        assigned = set()
        current = {}

        for obj in objs:
            c = tuple(obj.get("center", bbox_center(obj["bbox"])))
            best_tid = None
            best_cost = None

            for tid, st in active.items():
                if tid in assigned:
                    continue
                d = dist(c, st["center"])
                if d > fallback_max_dist:
                    continue
                cost = d / max(30.0, math.sqrt(max(1.0, bbox_area(obj["bbox"]))))
                if best_cost is None or cost < best_cost:
                    best_tid = tid
                    best_cost = cost

            if best_tid is None:
                tid = next_id
                next_id += 1
            else:
                tid = best_tid

            assigned.add(tid)
            current[tid] = {"center": c, "missed": 0}
            item = dict(obj)
            item["track_id"] = tid
            item["frame_id"] = fr.get("frame_id")
            item["time_sec"] = fr.get("time_sec")
            item["track_source"] = "custom_greedy"
            history[tid].append(item)

        for tid, st in active.items():
            if tid not in current:
                missed = st.get("missed", 0) + 1
                if missed <= 2:
                    current[tid] = {"center": st["center"], "missed": missed}
        active = current

    return dict(history)


def track_motion(track):
    pts = [tuple(x.get("center", bbox_center(x["bbox"]))) for x in track]
    if len(pts) < 2:
        return {"length": len(pts), "distance": 0.0, "displacement": 0.0, "mean_speed": 0.0, "direction_changes": 0}
    segs = [dist(pts[i - 1], pts[i]) for i in range(1, len(pts))]
    displacement = dist(pts[0], pts[-1])
    total_time = max(1e-6, float(track[-1]["time_sec"]) - float(track[0]["time_sec"]))
    direction_changes = 0
    for i in range(2, len(pts)):
        v1 = vec(pts[i - 2], pts[i - 1])
        v2 = vec(pts[i - 1], pts[i])
        if norm(v1) > 8 and norm(v2) > 8 and cosine(v1, v2) < 0.15:
            direction_changes += 1
    return {
        "length": len(pts),
        "distance": round(float(sum(segs)), 3),
        "displacement": round(float(displacement), 3),
        "mean_speed": round(float(sum(segs) / total_time), 3),
        "direction_changes": direction_changes,
        "first_frame": track[0]["frame_id"],
        "last_frame": track[-1]["frame_id"],
        "first_time": track[0]["time_sec"],
        "last_time": track[-1]["time_sec"],
        "track_source": track[0].get("track_source", "unknown"),
    }


def window_iter(frames, window_seconds=WINDOW_SECONDS, stride_seconds=WINDOW_STRIDE_SECONDS):
    if not frames:
        return []
    start = frames[0].get("time_sec", 0.0)
    end = frames[-1].get("time_sec", start)
    out = []
    t = start
    while t <= end + 1e-6:
        w_frames = [fr for fr in frames if t <= fr.get("time_sec", 0.0) < t + window_seconds]
        if w_frames:
            out.append((round(float(t), 3), round(float(t + window_seconds), 3), w_frames))
        t += stride_seconds
    return out


def compute_sidewalk_riding_score(frames):
    evidence = []
    tw_frame_count = 0
    rider_frame_count = 0
    sidewalk_hits = 0
    candidate_hits = 0
    road_hits = 0
    scores = []

    for fr in frames:
        if fr.get("two_wheelers"):
            tw_frame_count += 1
        if fr.get("rider_pairs"):
            rider_frame_count += 1

        for ev in fr.get("sidewalk_evidence", []):
            score = float(ev.get("sidewalk_score", 0.0))
            status = ev.get("sidewalk_status", "unknown")
            rider_bonus = 1.0 if ev.get("has_rider_pair") else 0.72
            final_score = min(1.0, score * rider_bonus + float(ev.get("best_rider_pair_score", 0.0)) * 0.16)
            item = {
                "frame_id": fr.get("frame_id"),
                "time_sec": fr.get("time_sec"),
                "status": status,
                "score": round(float(final_score), 4),
                "has_rider_pair": ev.get("has_rider_pair"),
                "best_rider_pair_score": ev.get("best_rider_pair_score"),
                "bbox": ev.get("bbox"),
                "two_wheeler_class": ev.get("two_wheeler_class"),
                "seg_sidewalk_ratio": ev.get("seg_sidewalk_ratio"),
                "band_sidewalk_ratio": ev.get("band_sidewalk_ratio"),
                "world_sidewalk_bbox_score": ev.get("world_sidewalk_bbox_score"),
                "road_score": ev.get("road_score"),
            }
            evidence.append(item)
            scores.append(final_score)
            if status == "sidewalk" and final_score >= SIDEWALK_RIDING_THRESHOLD:
                sidewalk_hits += 1
            elif status == "sidewalk_candidate" or final_score >= SIDEWALK_CANDIDATE_THRESHOLD:
                candidate_hits += 1
            elif status == "road":
                road_hits += 1

    total_ev = max(1, len(evidence))
    sidewalk_ratio = sidewalk_hits / total_ev
    candidate_ratio = (sidewalk_hits + candidate_hits) / total_ev
    rider_ratio = rider_frame_count / max(1, len(frames))
    tw_ratio = tw_frame_count / max(1, len(frames))
    mean_score = float(np.mean(scores)) if scores else 0.0
    max_score = float(np.max(scores)) if scores else 0.0

    # 이륜차 자체가 있어야 하며, rider pair가 있으면 강하게 반영
    final_score = 0.0
    final_score += min(0.42, max_score * 0.42)
    final_score += min(0.24, candidate_ratio * 0.24)
    final_score += min(0.18, sidewalk_ratio * 0.28)
    final_score += min(0.10, rider_ratio * 0.16)
    final_score += min(0.06, tw_ratio * 0.10)
    final_score = min(1.0, final_score)

    if not evidence:
        label = "no_two_wheeler_evidence"
        reason = "이륜차 bbox 또는 인도 근거가 없습니다."
    elif final_score >= SIDEWALK_RIDING_THRESHOLD:
        label = "sidewalk_riding"
        reason = "이륜차 기준점 또는 하단 band가 인도 영역에 반복적으로 위치했습니다."
    elif final_score >= SIDEWALK_CANDIDATE_THRESHOLD:
        label = "sidewalk_candidate"
        reason = "인도 주행 후보 근거는 있으나 반복성/확신이 부족합니다."
    elif road_hits > sidewalk_hits:
        label = "road_riding_or_unknown"
        reason = "도로 근거가 더 강하거나 인도 근거가 약합니다."
    else:
        label = "unknown"
        reason = "이륜차는 있으나 road/sidewalk 판단이 불명확합니다."

    return {
        "riding_label": label,
        "sidewalk_riding_score": round(float(final_score), 4),
        "is_sidewalk_riding": label == "sidewalk_riding",
        "tw_frame_count": tw_frame_count,
        "rider_frame_count": rider_frame_count,
        "sidewalk_hits": sidewalk_hits,
        "sidewalk_candidate_hits": candidate_hits,
        "road_hits": road_hits,
        "sidewalk_ratio": round(float(sidewalk_ratio), 4),
        "candidate_ratio": round(float(candidate_ratio), 4),
        "mean_evidence_score": round(float(mean_score), 4),
        "max_evidence_score": round(float(max_score), 4),
        "best_evidence": sorted(evidence, key=lambda x: x.get("score", 0), reverse=True)[:10],
        "reason": reason,
    }


def compute_helmet_score(frames):
    rider_frames = 0
    helmet_count = 0
    no_helmet_count = 0
    candidate_count = 0
    unknown_count = 0
    evidence = []
    scores = []

    for fr in frames:
        if fr.get("rider_pairs"):
            rider_frames += 1

        for ev in fr.get("helmet_evidence", []):
            status = ev.get("helmet_status", "unknown")
            conf = float(ev.get("helmet_confidence", 0.0))
            no_score = float(ev.get("no_helmet_score", 0.0))
            neutral = float(ev.get("neutral_head_score", 0.0))
            if status == "helmet":
                helmet_count += 1
            elif status == "no_helmet":
                no_helmet_count += 1
            elif status == "no_helmet_candidate":
                candidate_count += 1
            else:
                unknown_count += 1

            score = conf if status in ["no_helmet", "no_helmet_candidate"] else 0.0
            scores.append(score)
            item = dict(ev)
            item["time_sec"] = fr.get("time_sec")
            item["no_helmet_frame_score"] = round(float(score), 4)
            evidence.append(item)

    total_evidence = max(1, len(evidence))
    max_score = max(scores) if scores else 0.0
    no_ratio = no_helmet_count / total_evidence
    cand_ratio = (no_helmet_count + candidate_count) / total_evidence
    helmet_ratio = helmet_count / total_evidence
    unknown_ratio = unknown_count / total_evidence

    final_score = 0.0
    final_score += min(0.42, max_score * 0.42)
    final_score += min(0.24, no_ratio * 0.42)
    final_score += min(0.16, cand_ratio * 0.20)
    final_score += min(0.08, rider_frames / max(1, len(frames)) * 0.18)

    # rider가 반복적으로 보이는데 helmet positive가 거의 없고 head/no-helmet 후보가 있으면 absence 보조
    if rider_frames >= HELMET_UNKNOWN_MIN_RIDER_FRAMES and helmet_count == 0 and candidate_count > 0:
        final_score += 0.10
    final_score = min(1.0, final_score)

    if rider_frames == 0:
        label = "not_applicable"
        reason = "rider pair가 없어 헬멧 여부 판단 대상이 아닙니다."
    elif final_score >= NO_HELMET_THRESHOLD and no_helmet_count + candidate_count > 0 and helmet_ratio < 0.35:
        label = "no_helmet"
        reason = "rider head-zone에서 no-helmet/bare-head/head 근거가 helmet보다 강합니다."
    elif helmet_count > 0 and helmet_ratio >= max(0.25, cand_ratio):
        label = "helmet"
        reason = "rider head-zone에서 helmet 근거가 더 강합니다."
    else:
        label = "helmet_unknown"
        reason = "rider는 있으나 helmet/no-helmet 근거가 부족합니다. 전용 헬멧 모델 연결 시 개선됩니다."

    return {
        "helmet_label": label,
        "no_helmet_score": round(float(final_score), 4),
        "is_no_helmet": label == "no_helmet",
        "rider_frames": rider_frames,
        "helmet_count": helmet_count,
        "no_helmet_count": no_helmet_count,
        "no_helmet_candidate_count": candidate_count,
        "unknown_count": unknown_count,
        "helmet_ratio": round(float(helmet_ratio), 4),
        "no_helmet_ratio": round(float(no_ratio), 4),
        "candidate_ratio": round(float(cand_ratio), 4),
        "unknown_ratio": round(float(unknown_ratio), 4),
        "best_evidence": sorted(evidence, key=lambda x: x.get("no_helmet_frame_score", 0), reverse=True)[:12],
        "reason": reason,
    }


def compute_fight_window_score(w_frames):
    close_pairs = 0
    near_pairs = 0
    overlap_pairs = 0
    multi_person_frames = 0
    flow_vals = []
    pose_vals = []
    person_counts = []

    prev_by_order = None
    abrupt_motion = 0

    for fr in w_frames:
        persons = fr.get("persons", [])
        person_counts.append(len(persons))
        if len(persons) >= 2:
            multi_person_frames += 1

        for i in range(len(persons)):
            for j in range(i + 1, len(persons)):
                a, b = persons[i], persons[j]
                ca = a.get("center", bbox_center(a["bbox"]))
                cb = b.get("center", bbox_center(b["bbox"]))
                d = dist(ca, cb)
                if d <= FIGHT_CLOSE_DIST:
                    close_pairs += 1
                if d <= FIGHT_NEAR_DIST:
                    near_pairs += 1
                if bbox_iou(a["bbox"], b["bbox"]) >= FIGHT_OVERLAP_IOU or bbox_overlap_ratio(a["bbox"], b["bbox"]) > 0.04:
                    overlap_pairs += 1

        flow = fr.get("flow", {})
        flow_vals.append(float(flow.get("mean_mag", 0.0)) + float(flow.get("strong_ratio", 0.0)) * 8.0)
        pose_vals.append(float(fr.get("pose_agitation", 0.0)))

        centers = [tuple(p.get("center", bbox_center(p["bbox"]))) for p in persons]
        centers = sorted(centers)
        if prev_by_order is not None and len(prev_by_order) == len(centers) and len(centers) >= 2:
            frame_move = sum(dist(prev_by_order[k], centers[k]) for k in range(len(centers))) / len(centers)
            if frame_move > 35:
                abrupt_motion += 1
        prev_by_order = centers

    n = max(1, len(w_frames))
    pair_norm = max(1, n * 2)

    close_score = min(1.0, close_pairs / pair_norm)
    near_score = min(1.0, near_pairs / (pair_norm * 1.5))
    overlap_score = min(1.0, overlap_pairs / max(1, n))
    multi_score = multi_person_frames / n
    flow_score = min(1.0, (float(np.mean(flow_vals)) if flow_vals else 0.0) / 6.0)
    pose_score = min(1.0, float(np.mean(pose_vals)) if pose_vals else 0.0)
    abrupt_score = min(1.0, abrupt_motion / max(1, n - 1))

    score = (
        close_score * 0.22
        + near_score * 0.12
        + overlap_score * 0.22
        + multi_score * 0.12
        + flow_score * 0.13
        + pose_score * 0.11
        + abrupt_score * 0.08
    )
    return {
        "window_score": round(float(min(1.0, score)), 4),
        "close_pairs": close_pairs,
        "near_pairs": near_pairs,
        "overlap_pairs": overlap_pairs,
        "multi_person_frames": multi_person_frames,
        "mean_person_count": round(float(np.mean(person_counts)) if person_counts else 0.0, 3),
        "flow_score": round(float(flow_score), 4),
        "pose_score": round(float(pose_score), 4),
        "abrupt_score": round(float(abrupt_score), 4),
    }


def compute_fight_score(frames):
    windows = []
    for s, e, w_frames in window_iter(frames):
        item = compute_fight_window_score(w_frames)
        item["start_sec"] = s
        item["end_sec"] = e
        item["frame_count"] = len(w_frames)
        windows.append(item)

    best = sorted(windows, key=lambda x: x["window_score"], reverse=True)[0] if windows else None
    max_score = best["window_score"] if best else 0.0
    high_count = sum(1 for w in windows if w["window_score"] >= FIGHT_CANDIDATE_THRESHOLD)
    persistence = high_count / max(1, len(windows))
    final_score = min(1.0, max_score * 0.78 + persistence * 0.22)

    if final_score >= FIGHT_THRESHOLD:
        label = "fight"
        reason = "사람 간 근접/겹침과 강한 움직임이 같은 window에서 동시에 나타났습니다."
    elif final_score >= FIGHT_CANDIDATE_THRESHOLD:
        label = "fight_candidate"
        reason = "싸움 후보 움직임은 있으나 지속성 또는 겹침 근거가 부족합니다."
    else:
        label = "not_fight"
        reason = "싸움 조건을 만족하는 window가 없습니다."

    return {
        "fight_label": label,
        "fight_score": round(float(final_score), 4),
        "is_fight": label == "fight",
        "best_fight_window": best,
        "high_window_count": high_count,
        "window_count": len(windows),
        "windows": windows[:30],
        "reason": reason,
    }


def common_track_times(track_a, track_b):
    by_a = {x["frame_id"]: x for x in track_a}
    by_b = {x["frame_id"]: x for x in track_b}
    common = sorted(set(by_a) & set(by_b))
    return [(f, by_a[f], by_b[f]) for f in common]


def compute_pair_stalking(track_a, track_b):
    common = common_track_times(track_a, track_b)
    if len(common) < MIN_TRACK_LEN_FOR_STALKING:
        return None

    follow_events = 0
    same_direction_events = 0
    distance_values = []
    behind_values = []
    evidence_frames = []

    prev_a = None
    prev_b = None

    for f, a, b in common:
        ca = a.get("center", bbox_center(a["bbox"]))
        cb = b.get("center", bbox_center(b["bbox"]))
        d = dist(ca, cb)
        distance_values.append(d)

        if prev_a is not None and prev_b is not None:
            va = vec(prev_a.get("center", bbox_center(prev_a["bbox"])), ca)
            vb = vec(prev_b.get("center", bbox_center(prev_b["bbox"])), cb)
            if norm(va) > 5 and norm(vb) > 5:
                cosv = cosine(va, vb)
                if cosv >= FOLLOW_DIRECTION_COS_THRESHOLD:
                    same_direction_events += 1

                # A가 B를 따라가는지 또는 B가 A를 따라가는지 둘 다 검사
                # movement 방향 뒤쪽에 있는 사람이 follower라고 판단
                vmean = [va[0] + vb[0], va[1] + vb[1]]
                if norm(vmean) > 1e-6:
                    # cb-ca를 이동방향에 투영. 양수면 b가 a보다 앞쪽, a가 뒤따름
                    proj_b_a = ((cb[0] - ca[0]) * vmean[0] + (cb[1] - ca[1]) * vmean[1]) / norm(vmean)
                    proj_a_b = -proj_b_a
                    behind_proj = max(proj_b_a, proj_a_b)
                    behind_values.append(behind_proj)

                    if (
                        FOLLOW_DISTANCE_MIN <= d <= FOLLOW_DISTANCE_MAX
                        and cosv >= FOLLOW_DIRECTION_COS_THRESHOLD
                        and behind_proj >= FOLLOW_BEHIND_PROJECTION_MIN
                    ):
                        follow_events += 1
                        evidence_frames.append({
                            "frame_id": f,
                            "time_sec": a.get("time_sec"),
                            "distance": round(float(d), 2),
                            "same_direction_cos": round(float(cosv), 4),
                            "behind_projection": round(float(behind_proj), 2),
                        })

        prev_a = a
        prev_b = b

    event_ratio = follow_events / max(1, len(common) - 1)
    same_dir_ratio = same_direction_events / max(1, len(common) - 1)
    dist_std = float(np.std(distance_values)) if distance_values else 9999.0
    stable_score = max(0.0, 1.0 - dist_std / FOLLOW_DISTANCE_STABILITY)
    duration_sec = max(0.0, float(common[-1][1]["time_sec"]) - float(common[0][1]["time_sec"]))

    score = (
        event_ratio * 0.42
        + same_dir_ratio * 0.22
        + stable_score * 0.18
        + min(1.0, duration_sec / 8.0) * 0.12
        + min(1.0, len(common) / 16.0) * 0.06
    )

    return {
        "score": round(float(min(1.0, score)), 4),
        "common_frames": len(common),
        "follow_events": follow_events,
        "same_direction_events": same_direction_events,
        "event_ratio": round(float(event_ratio), 4),
        "same_direction_ratio": round(float(same_dir_ratio), 4),
        "distance_mean": round(float(np.mean(distance_values)), 3) if distance_values else None,
        "distance_std": round(float(dist_std), 3) if distance_values else None,
        "stable_score": round(float(stable_score), 4),
        "duration_sec": round(float(duration_sec), 3),
        "evidence_frames": evidence_frames[:12],
    }


def compute_stalking_score(frames, tracks):
    valid = {tid: tr for tid, tr in tracks.items() if len(tr) >= MIN_TRACK_LEN_FOR_STALKING}
    candidates = []

    tids = list(valid.keys())
    for i in range(len(tids)):
        for j in range(i + 1, len(tids)):
            a, b = tids[i], tids[j]
            pair = compute_pair_stalking(valid[a], valid[b])
            if pair is None:
                continue
            pair["pair"] = [a, b]
            candidates.append(pair)

    if not candidates:
        return {
            "stalking_label": "not_stalking",
            "stalking_score": 0.0,
            "is_stalking": False,
            "best_pair": None,
            "candidate_pairs": [],
            "reason": "스토킹 조건을 비교할 만큼 긴 사람 track pair가 없습니다.",
        }

    best = sorted(candidates, key=lambda x: x["score"], reverse=True)[0]
    score = best["score"]

    if score >= STALKING_THRESHOLD:
        label = "stalking"
        reason = "두 사람 track이 같은 방향으로 이동하고 일정 거리/뒤따름 구조가 유지됩니다."
    elif score >= STALKING_CANDIDATE_THRESHOLD:
        label = "stalking_candidate"
        reason = "따라가는 후보는 있으나 지속성 또는 거리 안정성이 부족합니다."
    else:
        label = "not_stalking"
        reason = "스토킹 조건 점수가 낮습니다."

    return {
        "stalking_label": label,
        "stalking_score": round(float(score), 4),
        "is_stalking": label == "stalking",
        "best_pair": best,
        "candidate_pairs": sorted(candidates, key=lambda x: x["score"], reverse=True)[:10],
        "reason": reason,
    }



# ------------------------------------------------------------
# DD 통합 침입 reasoning
#   기존 침입 파이썬 파일의 핵심 구조를 외부 import 없이 현재 feature에 맞게 재구성한다.
#   핵심: gate/fence/wall/barrier 후보 + 사람 foot_point track + boundary line crossing
# ------------------------------------------------------------
def _label_has_any(label, hints):
    label = normalize_label(label)
    return any(h in label for h in hints)


def is_intrusion_structure_label(label):
    return _label_has_any(label, INTRUSION_STRUCTURE_HINTS)


def intrusion_subtype_from_label(label):
    label = normalize_label(label)
    if _label_has_any(label, GATE_STRUCTURE_HINTS):
        return "gate_intrusion"
    if _label_has_any(label, FENCE_STRUCTURE_HINTS):
        return "fence_or_railing_intrusion"
    return "structure_intrusion"


def _pt(obj, prefer="bottom_center"):
    if prefer in obj:
        return obj[prefer]
    if prefer == "foot_point" and "bottom_center" in obj:
        return obj["bottom_center"]
    if "bbox" in obj:
        return bbox_bottom_center(obj["bbox"]) if prefer in ["bottom_center", "foot_point"] else bbox_center(obj["bbox"])
    return obj.get("center", [0.0, 0.0])


def _clip_point(p, frame_w, frame_h):
    return [
        int(clamp(round(float(p[0])), 0, max(0, frame_w - 1))),
        int(clamp(round(float(p[1])), 0, max(0, frame_h - 1))),
    ]


def _normalize_vec(v):
    n = norm(v)
    if n < 1e-6:
        return [0.0, 0.0]
    return [float(v[0]) / n, float(v[1]) / n]


def _side_of_line(point, p1, p2):
    x, y = float(point[0]), float(point[1])
    x1, y1 = float(p1[0]), float(p1[1])
    x2, y2 = float(p2[0]), float(p2[1])
    return (x2 - x1) * (y - y1) - (y2 - y1) * (x - x1)


def _median_point(points):
    if not points:
        return [0.0, 0.0]
    xs = [float(p[0]) for p in points]
    ys = [float(p[1]) for p in points]
    return [round(float(np.median(xs)), 2), round(float(np.median(ys)), 2)]


def _smooth_track_for_intrusion(track, window=3):
    if len(track) < 3:
        return list(track)
    half = window // 2
    out = []
    for i, item in enumerate(track):
        s = max(0, i - half)
        e = min(len(track), i + half + 1)
        ni = dict(item)
        ni["intrusion_point"] = _median_point([_pt(x, "bottom_center") for x in track[s:e]])
        out.append(ni)
    return out


def _track_intrusion_summary(track):
    if not track:
        return {"length": 0, "movement_px": 0.0}
    pts = [_pt(x, "bottom_center") for x in track]
    k = max(2, len(pts) // 5)
    first = _median_point(pts[:k])
    last = _median_point(pts[-k:])
    move = dist(first, last)
    return {
        "length": len(track),
        "first": first,
        "last": last,
        "movement_px": round(float(move), 2),
        "first_frame": track[0].get("frame_id"),
        "last_frame": track[-1].get("frame_id"),
        "first_time": track[0].get("time_sec"),
        "last_time": track[-1].get("time_sec"),
    }


def _select_intrusion_tracks(person_tracks):
    out = []
    for tid, tr in person_tracks.items():
        if len(tr) < INTRUSION_MIN_TRACK_LEN:
            continue
        sm = _smooth_track_for_intrusion(tr, window=3)
        summary = _track_intrusion_summary(sm)
        if summary.get("movement_px", 0) < INTRUSION_MIN_MOVEMENT_PX:
            continue
        out.append({"track_id": str(tid), "points": sm, **summary})
    return sorted(out, key=lambda x: (x["movement_px"], x["length"]), reverse=True)


def _flatten_intrusion_structures(frames):
    objects = []
    for fr in frames:
        for obj in fr.get("intrusion_structure_objects", []):
            label = obj.get("class") or obj.get("label") or ""
            if not is_intrusion_structure_label(label):
                continue
            item = dict(obj)
            item["frame_id"] = fr.get("frame_id")
            item["time_sec"] = fr.get("time_sec")
            item["label"] = normalize_label(label)
            objects.append(item)
    return objects


def _object_track_min_distance(obj, tracks):
    best_dist = 999999.0
    best_track = None
    best_point = None
    bbox = obj.get("bbox")
    if not bbox:
        return best_dist, best_track, best_point
    for tr in tracks:
        for p in tr["points"]:
            pt = p.get("intrusion_point") or _pt(p, "bottom_center")
            d = point_to_bbox_distance(pt, bbox)
            if d < best_dist:
                best_dist = d
                best_track = tr
                best_point = p
    return round(float(best_dist), 2), best_track, best_point


def _select_intrusion_structures(frames, track_candidates):
    objects = _flatten_intrusion_structures(frames)
    candidates = []
    frame_area = None
    if frames:
        # meta가 없어도 bbox scale 기반으로 충분히 동작한다.
        max_x = 0
        max_y = 0
        for fr in frames:
            for k in ["persons", "two_wheelers", "world_objects"]:
                for o in fr.get(k, []):
                    if "bbox" in o:
                        max_x = max(max_x, o["bbox"][2])
                        max_y = max(max_y, o["bbox"][3])
        frame_area = max(1, max_x * max_y)

    for obj in objects:
        bbox = obj.get("bbox")
        if not bbox or bbox_area(bbox) < 250:
            continue
        min_dist, best_track, best_point = _object_track_min_distance(obj, track_candidates)
        if min_dist > INTRUSION_MAX_STRUCTURE_DISTANCE:
            # 멀리 있는 구조물은 후보로는 남기지 않는다.
            continue
        conf = float(obj.get("confidence", 0.0))
        area = bbox_area(bbox)
        shape_score = min(1.0, area / max(60000.0, frame_area * 0.04 if frame_area else 60000.0))
        proximity_score = 1.0 / (1.0 + min_dist / 140.0)
        x1, y1, x2, y2 = bbox
        w, h = max(1, x2 - x1), max(1, y2 - y1)
        aspect = max(w / h, h / w)
        aspect_score = min(1.0, aspect / 4.0)
        label_bonus = 1.0 if intrusion_subtype_from_label(obj.get("label")) != "structure_intrusion" else 0.82
        score = conf * 0.36 + proximity_score * 0.34 + shape_score * 0.10 + aspect_score * 0.08 + label_bonus * 0.12
        candidates.append({
            "object": obj,
            "score": round(float(score), 4),
            "distance_to_track": min_dist,
            "distance_score": round(float(proximity_score), 4),
            "shape_score": round(float(shape_score), 4),
            "aspect_score": round(float(aspect_score), 4),
            "nearest_track_id": best_track.get("track_id") if best_track else None,
            "nearest_track_frame": best_point.get("frame_id") if best_point else None,
        })
    return sorted(candidates, key=lambda x: x["score"], reverse=True)[:12]


def _infer_intrusion_direction(track_candidates):
    if not track_candidates:
        return [0.0, 1.0]
    tr = track_candidates[0]
    v = [tr["last"][0] - tr["first"][0], tr["last"][1] - tr["first"][1]]
    nv = _normalize_vec(v)
    return nv if nv != [0.0, 0.0] else [0.0, 1.0]


def _nearest_track_item_to_bbox(track_points, bbox):
    best = None
    best_dist = 999999.0
    for p in track_points:
        pt = p.get("intrusion_point") or _pt(p, "bottom_center")
        d = point_to_bbox_distance(pt, bbox)
        if d < best_dist:
            best_dist = d
            best = p
    return best, round(float(best_dist), 2)


def _build_intrusion_line(struct_obj, track_points, frame_w, frame_h, direction, center_ratio=0.72):
    bbox = struct_obj["bbox"]
    x1, y1, x2, y2 = bbox
    bw = max(1, x2 - x1)
    bh = max(1, y2 - y1)
    nearest, nd = _nearest_track_item_to_bbox(track_points, bbox)

    # gate는 하단, fence/wall은 여러 높이 후보를 사용한다.
    sx = (x1 + x2) / 2
    sy = y1 + bh * center_ratio
    if nearest is not None:
        npnt = nearest.get("intrusion_point") or _pt(nearest, "bottom_center")
        cx = sx * 0.72 + npnt[0] * 0.28
        cy = sy * 0.82 + npnt[1] * 0.18
    else:
        cx, cy = sx, sy

    # nearest 전후 local movement로 line 방향 보정
    idx = 0
    if nearest is not None:
        for i, p in enumerate(track_points):
            if p.get("frame_id") == nearest.get("frame_id"):
                idx = i
                break
    s = max(0, idx - 2)
    e = min(len(track_points) - 1, idx + 2)
    before = track_points[s].get("intrusion_point") or _pt(track_points[s], "bottom_center")
    after = track_points[e].get("intrusion_point") or _pt(track_points[e], "bottom_center")
    local_vec = vec(before, after)
    v = _normalize_vec(local_vec) if dist(before, after) >= 10 else _normalize_vec(direction)
    if v == [0.0, 0.0]:
        v = _normalize_vec(direction)
    if v == [0.0, 0.0]:
        v = [0.0, 1.0]

    n = [-v[1], v[0]]
    if bw >= bh * 1.2:
        n = _normalize_vec([n[0] * 0.35 + 0.65, n[1] * 0.35])
    if n == [0.0, 0.0]:
        n = [1.0, 0.0]

    line_len = int(max(180, min(620, max(bw * 1.20, bh * 1.15, 240))))
    p1 = _clip_point([cx - n[0] * line_len / 2, cy - n[1] * line_len / 2], frame_w, frame_h)
    p2 = _clip_point([cx + n[0] * line_len / 2, cy + n[1] * line_len / 2], frame_w, frame_h)
    return {
        "boundary_line": [p1, p2],
        "line_center": [round(float(cx), 2), round(float(cy), 2)],
        "line_direction_vector": [round(float(v[0]), 6), round(float(v[1]), 6)],
        "line_length": line_len,
        "center_ratio": center_ratio,
        "nearest_track_frame": nearest.get("frame_id") if nearest else None,
        "nearest_track_distance": nd,
        "bbox": bbox,
    }


def _count_forward_side(points, start_idx, target_side, line, margin):
    p1, p2 = line
    c = 0
    for item in points[start_idx:]:
        pt = item.get("intrusion_point") or _pt(item, "bottom_center")
        sv = _side_of_line(pt, p1, p2)
        if target_side > 0 and sv > margin:
            c += 1
        elif target_side < 0 and sv < -margin:
            c += 1
        else:
            break
    return c


def _count_backward_side(points, end_idx, target_side, line, margin):
    p1, p2 = line
    c = 0
    for item in reversed(points[:end_idx + 1]):
        pt = item.get("intrusion_point") or _pt(item, "bottom_center")
        sv = _side_of_line(pt, p1, p2)
        if target_side > 0 and sv > margin:
            c += 1
        elif target_side < 0 and sv < -margin:
            c += 1
        else:
            break
    return c


def _find_same_track_intrusion_crossing(track_points, line_info, direction, struct_bbox):
    line = line_info["boundary_line"]
    p1, p2 = line
    candidates = []
    for after_idx, after in enumerate(track_points):
        apt = after.get("intrusion_point") or _pt(after, "bottom_center")
        after_sv = _side_of_line(apt, p1, p2)
        if abs(after_sv) <= INTRUSION_LINE_MARGIN:
            continue
        after_side = 1 if after_sv > 0 else -1
        after_streak = _count_forward_side(track_points, after_idx, after_side, line, INTRUSION_LINE_MARGIN)
        if after_streak < INTRUSION_MIN_AFTER_STREAK:
            continue
        before_side = -after_side
        for before_idx in range(0, after_idx):
            before = track_points[before_idx]
            bpt = before.get("intrusion_point") or _pt(before, "bottom_center")
            before_sv = _side_of_line(bpt, p1, p2)
            if before_side > 0 and before_sv <= INTRUSION_LINE_MARGIN:
                continue
            if before_side < 0 and before_sv >= -INTRUSION_LINE_MARGIN:
                continue
            before_streak = _count_backward_side(track_points, before_idx, before_side, line, INTRUSION_LINE_MARGIN)
            if before_streak < INTRUSION_MIN_BEFORE_STREAK:
                continue
            frame_gap = int(after.get("frame_id", 0)) - int(before.get("frame_id", 0))
            if frame_gap < INTRUSION_MIN_FRAME_GAP:
                continue
            movement = dist(bpt, apt)
            if movement < INTRUSION_MIN_MOVEMENT_PX * 0.65:
                continue
            mid = [(bpt[0] + apt[0]) / 2, (bpt[1] + apt[1]) / 2]
            dist_to_struct = point_to_bbox_distance(mid, struct_bbox)
            if dist_to_struct > INTRUSION_MAX_STRUCTURE_DISTANCE:
                continue
            mv = vec(bpt, apt)
            dir_cos = cosine(mv, direction)
            # 자동 방향은 영상마다 반대일 수 있으므로 너무 강하게 자르지 않는다.
            if dir_cos < -0.35:
                continue
            candidates.append({
                "type": "same_track_crossing",
                "before": {**before, "point": [round(float(bpt[0]), 2), round(float(bpt[1]), 2)], "line_side_value": round(float(before_sv), 3), "line_side": before_side},
                "after": {**after, "point": [round(float(apt[0]), 2), round(float(apt[1]), 2)], "line_side_value": round(float(after_sv), 3), "line_side": after_side},
                "side_transition": f"{before_side}_to_{after_side}",
                "frame_gap": frame_gap,
                "duration_sec": round(float(after.get("time_sec", 0) - before.get("time_sec", 0)), 3),
                "movement_px": round(float(movement), 2),
                "before_streak": before_streak,
                "after_streak": after_streak,
                "direction_cos": round(float(dir_cos), 4),
                "cross_midpoint": [round(float(mid[0]), 2), round(float(mid[1]), 2)],
                "cross_midpoint_distance_to_structure": round(float(dist_to_struct), 2),
            })
    if not candidates:
        return None
    return sorted(candidates, key=lambda x: (x["movement_px"], x["after_streak"], x["before_streak"], x["direction_cos"]), reverse=True)[0]


def _track_side_ratio(points, line, margin=None):
    if margin is None:
        margin = INTRUSION_LINE_MARGIN
    pos = neg = near = 0
    for item in points:
        pt = item.get("intrusion_point") or _pt(item, "bottom_center")
        sv = _side_of_line(pt, line[0], line[1])
        if sv > margin:
            pos += 1
        elif sv < -margin:
            neg += 1
        else:
            near += 1
    total = max(1, len(points))
    return {
        "positive_ratio": pos / total,
        "negative_ratio": neg / total,
        "near_ratio": near / total,
        "dominant_side": 1 if pos >= neg else -1,
        "dominant_ratio": max(pos, neg) / total,
    }


def _find_id_switch_intrusion_crossing(track_candidates, line_info, direction, struct_bbox):
    line = line_info["boundary_line"]
    out = []
    for prev in track_candidates:
        for nxt in track_candidates:
            if prev["track_id"] == nxt["track_id"]:
                continue
            prev_last = prev["points"][-1]
            next_first = nxt["points"][0]
            time_gap = float(next_first.get("time_sec", 0)) - float(prev_last.get("time_sec", 0))
            frame_gap = int(next_first.get("frame_id", 0)) - int(prev_last.get("frame_id", 0))
            if frame_gap <= 0 or time_gap < 0 or time_gap > INTRUSION_MAX_ID_SWITCH_GAP_SEC:
                continue
            ppt = prev_last.get("intrusion_point") or _pt(prev_last, "bottom_center")
            npt = next_first.get("intrusion_point") or _pt(next_first, "bottom_center")
            d = dist(ppt, npt)
            if d > INTRUSION_MAX_ID_SWITCH_DIST:
                continue
            prev_ratio = _track_side_ratio(prev["points"], line)
            next_ratio = _track_side_ratio(nxt["points"], line)
            if prev_ratio["dominant_ratio"] < 0.58 or next_ratio["dominant_ratio"] < 0.58:
                continue
            if prev_ratio["dominant_side"] == next_ratio["dominant_side"]:
                continue
            mid = [(ppt[0] + npt[0]) / 2, (ppt[1] + npt[1]) / 2]
            dist_to_struct = point_to_bbox_distance(mid, struct_bbox)
            if dist_to_struct > INTRUSION_MAX_STRUCTURE_DISTANCE:
                continue
            dir_cos = cosine(vec(ppt, npt), direction)
            out.append({
                "type": "id_switch_crossing",
                "prev_track_id": prev["track_id"],
                "next_track_id": nxt["track_id"],
                "before": {**prev_last, "point": [round(float(ppt[0]), 2), round(float(ppt[1]), 2)], "line_side": prev_ratio["dominant_side"]},
                "after": {**next_first, "point": [round(float(npt[0]), 2), round(float(npt[1]), 2)], "line_side": next_ratio["dominant_side"]},
                "side_transition": f"{prev_ratio['dominant_side']}_to_{next_ratio['dominant_side']}",
                "frame_gap": frame_gap,
                "duration_sec": round(float(time_gap), 3),
                "movement_px": round(float(d), 2),
                "direction_cos": round(float(dir_cos), 4),
                "cross_midpoint": [round(float(mid[0]), 2), round(float(mid[1]), 2)],
                "cross_midpoint_distance_to_structure": round(float(dist_to_struct), 2),
                "prev_dominant_ratio": round(float(prev_ratio["dominant_ratio"]), 3),
                "next_dominant_ratio": round(float(next_ratio["dominant_ratio"]), 3),
            })
    if not out:
        return None
    return sorted(out, key=lambda x: (x["movement_px"], -x["frame_gap"], x["direction_cos"]), reverse=True)[0]


def compute_integrated_intrusion_score(frames, person_tracks):
    if not RUN_INTRUSION_INTEGRATED:
        return {
            "intrusion_label": "not_run",
            "intrusion_score": 0.0,
            "detected_events": [],
            "reason": "RUN_INTRUSION_INTEGRATED=False 입니다.",
        }

    track_candidates = _select_intrusion_tracks(person_tracks)
    if not track_candidates:
        return {
            "intrusion_label": "no_intrusion",
            "intrusion_score": 0.0,
            "detected_events": [],
            "structure_candidates": [],
            "reason": "충분히 이동한 사람 track이 없어 침입 crossing을 판단하지 않았습니다.",
        }

    structure_candidates = _select_intrusion_structures(frames, track_candidates)
    if not structure_candidates:
        return {
            "intrusion_label": "no_intrusion",
            "intrusion_score": 0.0,
            "detected_events": [],
            "track_candidates": [{"track_id": t["track_id"], "movement_px": t["movement_px"], "length": t["length"]} for t in track_candidates[:8]],
            "structure_candidates": [],
            "reason": "gate/fence/railing/barrier/wall 구조 후보가 사람 이동 경로 근처에서 충분히 검출되지 않았습니다.",
        }

    # 대략적 frame 크기 추정
    max_x = max_y = 1
    for fr in frames:
        for k in ["persons", "two_wheelers", "world_objects"]:
            for o in fr.get(k, []):
                if "bbox" in o:
                    max_x = max(max_x, int(o["bbox"][2]) + 1)
                    max_y = max(max_y, int(o["bbox"][3]) + 1)
    frame_w, frame_h = max_x, max_y
    direction = _infer_intrusion_direction(track_candidates)

    all_results = []
    for sc in structure_candidates:
        obj = sc["object"]
        subtype = intrusion_subtype_from_label(obj.get("label"))
        # gate는 하단 중심을 더 강하게, fence/wall은 여러 높이 후보를 탐색
        center_ratios = [0.84, 0.72, 0.60] if subtype == "gate_intrusion" else [0.32, 0.42, 0.52, 0.62, 0.72, 0.84]
        relevant_tracks = sorted(
            track_candidates,
            key=lambda t: min(point_to_bbox_distance((p.get("intrusion_point") or _pt(p, "bottom_center")), obj["bbox"]) for p in t["points"])
        )[:5]

        best_for_obj = None
        for tr in relevant_tracks:
            for ratio in center_ratios:
                line_info = _build_intrusion_line(obj, tr["points"], frame_w, frame_h, direction, center_ratio=ratio)
                ev = _find_same_track_intrusion_crossing(tr["points"], line_info, direction, obj["bbox"])
                if ev is not None:
                    ev["track_id"] = tr["track_id"]
                    ev["center_ratio"] = ratio
                    move_score = min(1.0, ev["movement_px"] / 190.0)
                    streak_score = min(1.0, (ev.get("before_streak", 0) + ev.get("after_streak", 0)) / 8.0)
                    prox_score = max(0.0, 1.0 - ev.get("cross_midpoint_distance_to_structure", 999) / INTRUSION_MAX_STRUCTURE_DISTANCE)
                    dir_score = max(0.0, ev.get("direction_cos", 0.0))
                    score = sc["score"] * 0.28 + move_score * 0.25 + streak_score * 0.18 + prox_score * 0.20 + dir_score * 0.09
                    result = {
                        "event": ev,
                        "line_info": line_info,
                        "structure_candidate": sc,
                        "track": {"track_id": tr["track_id"], "movement_px": tr["movement_px"], "length": tr["length"]},
                        "event_subtype": subtype,
                        "score": round(float(score), 4),
                    }
                    if best_for_obj is None or result["score"] > best_for_obj["score"]:
                        best_for_obj = result

        # same-track이 없을 때 ID switch 보정
        if best_for_obj is None and relevant_tracks:
            tr0 = relevant_tracks[0]
            for ratio in center_ratios:
                line_info = _build_intrusion_line(obj, tr0["points"], frame_w, frame_h, direction, center_ratio=ratio)
                ev = _find_id_switch_intrusion_crossing(track_candidates, line_info, direction, obj["bbox"])
                if ev is not None:
                    move_score = min(1.0, ev["movement_px"] / 260.0)
                    prox_score = max(0.0, 1.0 - ev.get("cross_midpoint_distance_to_structure", 999) / INTRUSION_MAX_STRUCTURE_DISTANCE)
                    side_score = min(1.0, (ev.get("prev_dominant_ratio", 0) + ev.get("next_dominant_ratio", 0)) / 2.0)
                    dir_score = max(0.0, ev.get("direction_cos", 0.0))
                    score = sc["score"] * 0.28 + move_score * 0.22 + prox_score * 0.22 + side_score * 0.20 + dir_score * 0.08
                    result = {
                        "event": ev,
                        "line_info": line_info,
                        "structure_candidate": sc,
                        "track": {"track_id": f"{ev['prev_track_id']}->{ev['next_track_id']}", "movement_px": ev["movement_px"]},
                        "event_subtype": subtype,
                        "score": round(float(score), 4),
                    }
                    if best_for_obj is None or result["score"] > best_for_obj["score"]:
                        best_for_obj = result
        if best_for_obj is not None:
            all_results.append(best_for_obj)

    if not all_results:
        return {
            "intrusion_label": "no_intrusion",
            "intrusion_score": 0.0,
            "detected_events": [],
            "intrusion_direction_auto": direction,
            "structure_candidates": structure_candidates[:8],
            "track_candidates": [{"track_id": t["track_id"], "movement_px": t["movement_px"], "length": t["length"]} for t in track_candidates[:8]],
            "reason": "구조 후보는 찾았지만 사람 foot_point가 boundary line을 넘는 조건을 만족하지 않았습니다.",
        }

    best = sorted(all_results, key=lambda x: x["score"], reverse=True)[0]
    score = float(best["score"])
    if score >= INTRUSION_THRESHOLD:
        label = "intrusion"
        detected = True
        reason = "gate/fence/wall/barrier 구조물 근처에서 사람 foot_point track이 boundary line을 통과했습니다."
    elif score >= INTRUSION_CANDIDATE_THRESHOLD:
        label = "intrusion_candidate"
        detected = False
        reason = "침입 crossing 후보는 있으나 점수가 임계값보다 낮아 후보로만 표시합니다."
    else:
        label = "no_intrusion"
        detected = False
        reason = "침입 후보 점수가 낮습니다."

    confidence = min(0.93, round(0.50 + score * 0.58, 3)) if label == "intrusion" else round(score, 4)
    obj = best["structure_candidate"]["object"]
    detected_events = []
    if label in ["intrusion", "intrusion_candidate"]:
        detected_events.append({
            "event_class": "abnormal_path_intrusion",
            "event_subtype": best.get("event_subtype", "structure_intrusion"),
            "detected": detected,
            "confidence": confidence,
            "score": round(float(score), 4),
            "label": label,
            "scene_object": {
                "label": obj.get("label"),
                "confidence": obj.get("confidence"),
                "bbox": obj.get("bbox"),
                "frame_id": obj.get("frame_id"),
                "selection_score": best["structure_candidate"].get("score"),
                "distance_to_track": best["structure_candidate"].get("distance_to_track"),
            },
            "boundary_line": best.get("line_info"),
            "track": best.get("track"),
            "evidence": best.get("event"),
            "reason": reason,
            "source": "dd_integrated_intrusion_reasoning",
        })

    return {
        "intrusion_label": label,
        "intrusion_score": round(float(score), 4),
        "is_intrusion": label == "intrusion",
        "intrusion_direction_auto": direction,
        "detected_events": detected_events,
        "best_event": best,
        "candidate_results": sorted(all_results, key=lambda x: x["score"], reverse=True)[:10],
        "structure_candidates": structure_candidates[:8],
        "track_candidates": [{"track_id": t["track_id"], "movement_px": t["movement_px"], "length": t["length"]} for t in track_candidates[:8]],
        "reason": reason,
    }


def normalize_event(event_type, subtype, label, score, confidence=None, evidence=None, reason=None):
    if confidence is None:
        confidence = score
    return {
        "event_type": event_type,
        "event_subtype": subtype,
        "label": label,
        "score": round(float(score), 4),
        "confidence": round(float(confidence), 4),
        "evidence": evidence,
        "reason": reason,
    }


def fuse_events(fight, stalking, riding, helmet, intrusion_result=None):
    events = []

    # 싸움/스토킹은 같은 action 계열이라 둘 다 높으면 더 강한 쪽을 primary로 두되 후보는 남김
    if fight.get("fight_label") in ["fight", "fight_candidate"]:
        events.append(normalize_event(
            "abnormal_action",
            "fight",
            fight.get("fight_label"),
            fight.get("fight_score", 0.0),
            evidence=fight.get("best_fight_window"),
            reason=fight.get("reason"),
        ))

    if stalking.get("stalking_label") in ["stalking", "stalking_candidate"]:
        events.append(normalize_event(
            "abnormal_action",
            "stalking",
            stalking.get("stalking_label"),
            stalking.get("stalking_score", 0.0),
            evidence=stalking.get("best_pair"),
            reason=stalking.get("reason"),
        ))

    if riding.get("riding_label") in ["sidewalk_riding", "sidewalk_candidate"]:
        events.append(normalize_event(
            "traffic_violation",
            "sidewalk_two_wheeler",
            riding.get("riding_label"),
            riding.get("sidewalk_riding_score", 0.0),
            evidence=riding.get("best_evidence", [])[:5],
            reason=riding.get("reason"),
        ))

    if helmet.get("helmet_label") == "no_helmet":
        events.append(normalize_event(
            "traffic_violation",
            "no_helmet",
            helmet.get("helmet_label"),
            helmet.get("no_helmet_score", 0.0),
            evidence=helmet.get("best_evidence", [])[:5],
            reason=helmet.get("reason"),
        ))

    if intrusion_result and intrusion_result.get("intrusion_label") in ["intrusion", "intrusion_candidate"]:
        for ev in intrusion_result.get("detected_events", []):
            label = ev.get("label") or intrusion_result.get("intrusion_label")
            events.append(normalize_event(
                "abnormal_path_intrusion",
                ev.get("event_subtype", "intrusion"),
                label,
                intrusion_result.get("intrusion_score", ev.get("score", ev.get("confidence", 0.0))),
                confidence=ev.get("confidence", intrusion_result.get("intrusion_score", 0.0)),
                evidence=ev.get("evidence"),
                reason=ev.get("reason"),
            ))

    events = sorted(events, key=lambda x: x.get("confidence", 0), reverse=True)
    primary = events[0] if events and events[0]["confidence"] >= FINAL_MIN_EVENT_CONF else None

    final_tags = []
    if primary:
        final_tags.append(primary["event_subtype"])
    # 교통 위반 계열은 동시 발생 가능성이 높으므로 보조 태그로 유지
    for ev in events:
        if ev["event_subtype"] not in final_tags and ev["confidence"] >= FINAL_MIN_EVENT_CONF:
            final_tags.append(ev["event_subtype"])

    final_label = " + ".join(final_tags) if final_tags else "other"

    return {
        "final_label": final_label,
        "final_primary_event": primary,
        "candidate_events": events,
        "detected_event_count": len([e for e in events if e.get("confidence", 0) >= FINAL_MIN_EVENT_CONF]),
    }


In [7]:

# ============================================================
# Cell 6-B. EE enhanced reasoning override
#   - dd의 영상 인식/feature JSON 구조는 그대로 유지
#   - 전체 후보(fight/stalking/sidewalk/no_helmet/kickboard/intrusion)는 모두 유지
#   - 싸움/스토킹 feature만 track-pair box + sliding window 방식으로 강화
#   - final_label은 후보 중 가장 강한 하나만 선택
# ============================================================

def _ee_float(x, default=0.0):
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default


def _ee_bbox_union(a, b):
    ax1, ay1, ax2, ay2 = [float(v) for v in a]
    bx1, by1, bx2, by2 = [float(v) for v in b]
    return [
        int(round(min(ax1, bx1))),
        int(round(min(ay1, by1))),
        int(round(max(ax2, bx2))),
        int(round(max(ay2, by2))),
    ]


def _ee_bbox_size(b):
    x1, y1, x2, y2 = [float(v) for v in b]
    return max(1.0, x2 - x1), max(1.0, y2 - y1)


def _ee_frame_lookup(frames):
    return {fr.get("frame_id"): fr for fr in frames}


def _ee_obs_center(obj):
    try:
        return obj.get("center") or bbox_center(obj["bbox"])
    except Exception:
        return [0.0, 0.0]


def _ee_obs_bottom(obj):
    try:
        return obj.get("bottom_center") or bbox_bottom_center(obj["bbox"])
    except Exception:
        return _ee_obs_center(obj)


def _ee_common_obs(track_a, track_b, start_sec=None, end_sec=None):
    by_a = {x.get("frame_id"): x for x in track_a if x.get("frame_id") is not None}
    by_b = {x.get("frame_id"): x for x in track_b if x.get("frame_id") is not None}
    common_ids = sorted(set(by_a) & set(by_b))
    out = []
    for fid in common_ids:
        a, b = by_a[fid], by_b[fid]
        t = _ee_float(a.get("time_sec"), _ee_float(b.get("time_sec"), 0.0))
        if start_sec is not None and t < start_sec:
            continue
        if end_sec is not None and t >= end_sec:
            continue
        out.append((fid, t, a, b))
    return out


def _ee_track_slice(track, start_sec, end_sec):
    return [x for x in track if start_sec <= _ee_float(x.get("time_sec"), 0.0) < end_sec]


def _ee_pair_window_iter(frames, window_seconds, stride_seconds):
    if not frames:
        return []
    start = _ee_float(frames[0].get("time_sec"), 0.0)
    end = _ee_float(frames[-1].get("time_sec"), start)
    out = []
    t = start
    while t <= end + 1e-6:
        w_frames = [fr for fr in frames if t <= _ee_float(fr.get("time_sec"), 0.0) < t + window_seconds]
        if w_frames:
            out.append((round(float(t), 3), round(float(t + window_seconds), 3), w_frames))
        t += stride_seconds
    return out


def _ee_motion_stats(obs_common):
    if len(obs_common) < 2:
        return {"mean_speed": 0.0, "max_step": 0.0, "direction_changes": 0, "mean_relative_speed": 0.0}
    steps = []
    rel_steps = []
    dir_changes = 0
    prev_va = None
    prev_vb = None
    for i in range(1, len(obs_common)):
        _, t0, a0, b0 = obs_common[i - 1]
        _, t1, a1, b1 = obs_common[i]
        dt = max(1e-6, float(t1) - float(t0))
        ca0, ca1 = _ee_obs_center(a0), _ee_obs_center(a1)
        cb0, cb1 = _ee_obs_center(b0), _ee_obs_center(b1)
        va = vec(ca0, ca1)
        vb = vec(cb0, cb1)
        sa = norm(va) / dt
        sb = norm(vb) / dt
        steps.append((sa + sb) / 2.0)
        rel_steps.append(norm([va[0] - vb[0], va[1] - vb[1]]) / dt)
        if prev_va is not None and norm(prev_va) > 6 and norm(va) > 6 and cosine(prev_va, va) < 0.0:
            dir_changes += 1
        if prev_vb is not None and norm(prev_vb) > 6 and norm(vb) > 6 and cosine(prev_vb, vb) < 0.0:
            dir_changes += 1
        prev_va, prev_vb = va, vb
    return {
        "mean_speed": round(float(np.mean(steps)) if steps else 0.0, 4),
        "max_step": round(float(np.max(steps)) if steps else 0.0, 4),
        "direction_changes": int(dir_changes),
        "mean_relative_speed": round(float(np.mean(rel_steps)) if rel_steps else 0.0, 4),
    }


def _ee_pair_contact_stats(obs_common):
    distances = []
    norm_distances = []
    overlaps = []
    pair_boxes = []
    close_count = 0
    very_close_count = 0
    contact_count = 0

    for fid, t, a, b in obs_common:
        ba, bb = a.get("bbox"), b.get("bbox")
        if ba is None or bb is None:
            continue
        ca, cb = _ee_obs_center(a), _ee_obs_center(b)
        d = dist(ca, cb)
        _, ha = _ee_bbox_size(ba)
        _, hb = _ee_bbox_size(bb)
        scale = max(35.0, (ha + hb) / 2.0)
        nd = d / scale
        iou = bbox_iou(ba, bb)
        ov = max(iou, bbox_overlap_ratio(ba, bb))
        distances.append(d)
        norm_distances.append(nd)
        overlaps.append(ov)
        pbox = _ee_bbox_union(ba, bb)
        pair_boxes.append({
            "frame_id": fid,
            "time_sec": round(float(t), 3),
            "bbox": pbox,
            "distance": round(float(d), 2),
            "norm_distance": round(float(nd), 4),
            "overlap": round(float(ov), 5),
        })
        if nd < 1.20 or d <= FIGHT_NEAR_DIST:
            close_count += 1
        if nd < 0.72 or d <= FIGHT_CLOSE_DIST:
            very_close_count += 1
        if ov >= FIGHT_OVERLAP_IOU or nd < 0.55:
            contact_count += 1

    n = max(1, len(obs_common))
    return {
        "distance_mean": round(float(np.mean(distances)) if distances else 9999.0, 3),
        "distance_std": round(float(np.std(distances)) if distances else 9999.0, 3),
        "norm_distance_mean": round(float(np.mean(norm_distances)) if norm_distances else 9999.0, 4),
        "overlap_max": round(float(np.max(overlaps)) if overlaps else 0.0, 5),
        "close_ratio": round(float(close_count / n), 4),
        "very_close_ratio": round(float(very_close_count / n), 4),
        "contact_ratio": round(float(contact_count / n), 4),
        "pair_boxes": pair_boxes[:10],
    }


def _ee_frame_activity_score(w_frames):
    flow_vals = []
    pose_vals = []
    for fr in w_frames:
        flow = fr.get("flow", {}) or {}
        flow_vals.append(_ee_float(flow.get("mean_mag"), 0.0) + _ee_float(flow.get("strong_ratio"), 0.0) * 8.0)
        pose_vals.append(_ee_float(fr.get("pose_agitation"), 0.0))
    flow_score = min(1.0, (float(np.mean(flow_vals)) if flow_vals else 0.0) / 6.0)
    pose_score = min(1.0, float(np.mean(pose_vals)) if pose_vals else 0.0)
    return round(float(flow_score), 4), round(float(pose_score), 4)


def _ee_compute_pair_fight_window(pair, track_a, track_b, start_sec, end_sec, w_frames):
    obs = _ee_common_obs(track_a, track_b, start_sec, end_sec)
    if len(obs) < EE_PAIR_MIN_COMMON_FRAMES:
        return None

    contact = _ee_pair_contact_stats(obs)
    motion = _ee_motion_stats(obs)
    flow_score, pose_score = _ee_frame_activity_score(w_frames)

    # bbox fallback: 작은 CCTV 인물에서는 pose보다 pair distance/contact/motion을 더 신뢰
    close_score = min(1.0, contact["close_ratio"])
    contact_score = min(1.0, contact["contact_ratio"] * 1.10 + min(0.30, contact["overlap_max"] * 4.0))
    abrupt_score = min(1.0, motion["mean_relative_speed"] / 95.0 + motion["direction_changes"] / 6.0)
    speed_score = min(1.0, motion["mean_speed"] / 120.0)
    duration_score = min(1.0, len(obs) / max(2.0, len(w_frames)))

    score = (
        close_score * 0.20
        + contact_score * 0.26
        + abrupt_score * 0.22
        + speed_score * 0.10
        + flow_score * 0.08
        + pose_score * 0.08
        + duration_score * 0.06
    )

    # 두 사람이 정말 가까운 상태에서 큰 상대 움직임이면 싸움 가중
    if contact["very_close_ratio"] >= 0.35 and abrupt_score >= 0.28:
        score += 0.08
    if contact["contact_ratio"] >= 0.45 and (flow_score >= 0.25 or pose_score >= 0.20):
        score += 0.06

    return {
        "pair": [int(pair[0]), int(pair[1])],
        "start_sec": round(float(start_sec), 3),
        "end_sec": round(float(end_sec), 3),
        "frame_count": len(w_frames),
        "common_frames": len(obs),
        "window_score": round(float(min(1.0, score)), 4),
        "close_score": round(float(close_score), 4),
        "contact_score": round(float(contact_score), 4),
        "abrupt_score": round(float(abrupt_score), 4),
        "speed_score": round(float(speed_score), 4),
        "flow_score": flow_score,
        "pose_score": pose_score,
        "distance_mean": contact["distance_mean"],
        "norm_distance_mean": contact["norm_distance_mean"],
        "overlap_max": contact["overlap_max"],
        "close_ratio": contact["close_ratio"],
        "very_close_ratio": contact["very_close_ratio"],
        "contact_ratio": contact["contact_ratio"],
        "mean_relative_speed": motion["mean_relative_speed"],
        "mean_speed": motion["mean_speed"],
        "direction_changes": motion["direction_changes"],
        "pair_boxes": contact["pair_boxes"],
    }


def compute_fight_score(frames, tracks=None):
    """EE override: track-pair box + sliding-window fight scoring."""
    if tracks is None:
        tracks = assign_tracks_from_frame_objects(frames, object_key="persons", fallback_max_dist=170, min_conf=PERSON_MIN_CONF)

    valid = {tid: tr for tid, tr in (tracks or {}).items() if len(tr) >= 2}
    tids = list(valid.keys())
    windows = []

    if len(tids) >= 2:
        for s, e, w_frames in _ee_pair_window_iter(frames, EE_FIGHT_WINDOW_SECONDS, EE_FIGHT_WINDOW_STRIDE_SECONDS):
            for i in range(len(tids)):
                for j in range(i + 1, len(tids)):
                    item = _ee_compute_pair_fight_window((tids[i], tids[j]), valid[tids[i]], valid[tids[j]], s, e, w_frames)
                    if item is not None:
                        windows.append(item)

    # fallback: track id가 부족한 영상은 dd의 프레임 기반 window 점수를 그대로 사용
    if not windows:
        fallback_windows = []
        for s, e, w_frames in window_iter(frames, window_seconds=EE_FIGHT_WINDOW_SECONDS, stride_seconds=EE_FIGHT_WINDOW_STRIDE_SECONDS):
            item = compute_fight_window_score(w_frames)
            item["start_sec"] = s
            item["end_sec"] = e
            item["pair"] = None
            item["fallback_mode"] = "frame_order_without_track_pair"
            fallback_windows.append(item)
        windows = fallback_windows

    best = sorted(windows, key=lambda x: x.get("window_score", 0.0), reverse=True)[0] if windows else None
    max_score = _ee_float(best.get("window_score"), 0.0) if best else 0.0
    high_count = sum(1 for w in windows if _ee_float(w.get("window_score"), 0.0) >= FIGHT_CANDIDATE_THRESHOLD)
    strong_count = sum(1 for w in windows if _ee_float(w.get("window_score"), 0.0) >= FIGHT_THRESHOLD)
    persistence = high_count / max(1, len(windows))

    # 후반부에 갑자기 발생하는 싸움도 살리기 위해 max window 비중을 크게 둔다.
    final_score = min(1.0, max_score * 0.82 + persistence * 0.12 + min(0.06, strong_count * 0.03))

    if final_score >= FIGHT_THRESHOLD:
        label = "fight"
        reason = "track-pair window에서 근접/접촉/급격한 상대 움직임이 동시에 나타났습니다."
    elif final_score >= FIGHT_CANDIDATE_THRESHOLD:
        label = "fight_candidate"
        reason = "싸움 후보 window가 있으나 접촉/지속성 근거가 확정 기준에는 부족합니다."
    else:
        label = "not_fight"
        reason = "싸움 조건을 만족하는 pair-window가 없습니다."

    return {
        "fight_label": label,
        "fight_score": round(float(final_score), 4),
        "is_fight": label == "fight",
        "best_fight_window": best,
        "high_window_count": high_count,
        "strong_window_count": strong_count,
        "window_count": len(windows),
        "windows": sorted(windows, key=lambda x: x.get("window_score", 0.0), reverse=True)[:40],
        "reason": reason,
        "ee_method": "track_pair_box_sliding_window",
    }


def _ee_compute_pair_stalking_window(pair, track_a, track_b, start_sec, end_sec, w_frames):
    obs = _ee_common_obs(track_a, track_b, start_sec, end_sec)
    if len(obs) < EE_PAIR_MIN_COMMON_FRAMES:
        return None

    distances = []
    direction_hits = 0
    follow_hits = 0
    contact_hits = 0
    evidence = []
    pair_boxes = []

    prev = None
    cos_values = []
    behind_values = []

    for cur in obs:
        fid, t, a, b = cur
        ba, bb = a.get("bbox"), b.get("bbox")
        if ba is None or bb is None:
            continue
        ca, cb = _ee_obs_center(a), _ee_obs_center(b)
        d = dist(ca, cb)
        distances.append(d)
        ov = max(bbox_iou(ba, bb), bbox_overlap_ratio(ba, bb))
        _, ha = _ee_bbox_size(ba)
        _, hb = _ee_bbox_size(bb)
        nd = d / max(35.0, (ha + hb) / 2.0)
        if ov >= FIGHT_OVERLAP_IOU or nd < 0.62:
            contact_hits += 1
        pair_boxes.append({
            "frame_id": fid,
            "time_sec": round(float(t), 3),
            "bbox": _ee_bbox_union(ba, bb),
            "distance": round(float(d), 2),
        })

        if prev is not None:
            _, pt, pa, pb = prev
            va = vec(_ee_obs_center(pa), ca)
            vb = vec(_ee_obs_center(pb), cb)
            if norm(va) > 4 and norm(vb) > 4:
                cosv = cosine(va, vb)
                cos_values.append(cosv)
                if cosv >= FOLLOW_DIRECTION_COS_THRESHOLD:
                    direction_hits += 1

                # 평균 이동 방향 뒤쪽에 있는 사람이 follower 후보
                vmean = [va[0] + vb[0], va[1] + vb[1]]
                if norm(vmean) > 1e-6:
                    proj_b_a = ((cb[0] - ca[0]) * vmean[0] + (cb[1] - ca[1]) * vmean[1]) / norm(vmean)
                    behind_proj = abs(proj_b_a)
                    behind_values.append(behind_proj)
                    if (
                        FOLLOW_DISTANCE_MIN <= d <= FOLLOW_DISTANCE_MAX
                        and cosv >= FOLLOW_DIRECTION_COS_THRESHOLD
                        and behind_proj >= FOLLOW_BEHIND_PROJECTION_MIN
                    ):
                        follow_hits += 1
                        evidence.append({
                            "frame_id": fid,
                            "time_sec": round(float(t), 3),
                            "distance": round(float(d), 2),
                            "same_direction_cos": round(float(cosv), 4),
                            "behind_projection_abs": round(float(behind_proj), 2),
                        })
        prev = cur

    transitions = max(1, len(obs) - 1)
    direction_ratio = direction_hits / transitions
    follow_ratio = follow_hits / transitions
    contact_ratio = contact_hits / max(1, len(obs))
    dist_mean = float(np.mean(distances)) if distances else 9999.0
    dist_std = float(np.std(distances)) if distances else 9999.0
    stable_score = max(0.0, 1.0 - dist_std / FOLLOW_DISTANCE_STABILITY)
    duration_sec = max(0.0, float(obs[-1][1]) - float(obs[0][1])) if obs else 0.0
    distance_band_score = 1.0 if FOLLOW_DISTANCE_MIN <= dist_mean <= FOLLOW_DISTANCE_MAX else 0.0
    no_contact_score = max(0.0, 1.0 - contact_ratio * 1.8)

    # 스토킹은 접촉보다 '같은 방향 + 일정 거리 유지 + 뒤따름 + 비접촉'을 강하게 본다.
    score = (
        follow_ratio * 0.34
        + direction_ratio * 0.22
        + stable_score * 0.16
        + min(1.0, duration_sec / 8.0) * 0.10
        + min(1.0, len(obs) / 12.0) * 0.06
        + distance_band_score * 0.05
        + no_contact_score * 0.07
    )

    # 접촉/겹침이 많으면 스토킹보다 싸움 가능성이 크므로 감쇠
    if contact_ratio >= 0.30:
        score *= 0.62
    elif contact_ratio >= 0.18:
        score *= 0.78

    return {
        "pair": [int(pair[0]), int(pair[1])],
        "start_sec": round(float(start_sec), 3),
        "end_sec": round(float(end_sec), 3),
        "common_frames": len(obs),
        "frame_count": len(w_frames),
        "score": round(float(min(1.0, score)), 4),
        "follow_ratio": round(float(follow_ratio), 4),
        "same_direction_ratio": round(float(direction_ratio), 4),
        "distance_mean": round(float(dist_mean), 3),
        "distance_std": round(float(dist_std), 3),
        "stable_score": round(float(stable_score), 4),
        "contact_ratio": round(float(contact_ratio), 4),
        "duration_sec": round(float(duration_sec), 3),
        "evidence_frames": evidence[:12],
        "pair_boxes": pair_boxes[:10],
        "cos_mean": round(float(np.mean(cos_values)), 4) if cos_values else None,
        "behind_mean_abs": round(float(np.mean(behind_values)), 3) if behind_values else None,
    }


def compute_stalking_score(frames, tracks, fight_context=None):
    """EE override: stalking도 fight와 같은 track-pair box + sliding-window 구조 사용."""
    valid = {tid: tr for tid, tr in (tracks or {}).items() if len(tr) >= MIN_TRACK_LEN_FOR_STALKING}
    tids = list(valid.keys())
    candidates = []

    for s, e, w_frames in _ee_pair_window_iter(frames, EE_STALKING_WINDOW_SECONDS, EE_STALKING_WINDOW_STRIDE_SECONDS):
        for i in range(len(tids)):
            for j in range(i + 1, len(tids)):
                item = _ee_compute_pair_stalking_window((tids[i], tids[j]), valid[tids[i]], valid[tids[j]], s, e, w_frames)
                if item is not None:
                    candidates.append(item)

    if not candidates:
        return {
            "stalking_label": "not_stalking",
            "stalking_score": 0.0,
            "is_stalking": False,
            "best_pair": None,
            "candidate_pairs": [],
            "reason": "스토킹 조건을 비교할 만큼 긴 사람 track pair/window가 없습니다.",
            "ee_method": "track_pair_box_sliding_window",
        }

    candidates = sorted(candidates, key=lambda x: x.get("score", 0.0), reverse=True)
    best = candidates[0]
    max_score = _ee_float(best.get("score"), 0.0)
    high_count = sum(1 for c in candidates if _ee_float(c.get("score"), 0.0) >= STALKING_CANDIDATE_THRESHOLD)
    persistence = high_count / max(1, len(candidates))
    final_score = min(1.0, max_score * 0.84 + persistence * 0.16)

    # 싸움 window가 강하면 스토킹 점수를 낮춘다. 특히 접촉/급격한 움직임 근거가 있으면 더 낮춘다.
    if isinstance(fight_context, dict):
        fscore = _ee_float(fight_context.get("fight_score"), 0.0)
        fw = fight_context.get("best_fight_window") or {}
        contact_ratio = _ee_float(fw.get("contact_ratio"), 0.0)
        abrupt_score = _ee_float(fw.get("abrupt_score"), 0.0)
        if fscore >= FIGHT_THRESHOLD:
            final_score *= 0.58
        elif fscore >= FIGHT_CANDIDATE_THRESHOLD and (contact_ratio >= 0.20 or abrupt_score >= 0.35):
            final_score *= 0.72

    if final_score >= STALKING_THRESHOLD:
        label = "stalking"
        reason = "track-pair window에서 같은 방향 이동, 일정 거리 유지, 뒤따름 구조가 비접촉 상태로 지속됩니다."
    elif final_score >= STALKING_CANDIDATE_THRESHOLD:
        label = "stalking_candidate"
        reason = "스토킹 후보 window가 있으나 지속성/거리 안정성/뒤따름 근거가 확정 기준에는 부족합니다."
    else:
        label = "not_stalking"
        reason = "스토킹 조건을 만족하는 pair-window가 없습니다."

    return {
        "stalking_label": label,
        "stalking_score": round(float(final_score), 4),
        "is_stalking": label == "stalking",
        "best_pair": best,
        "candidate_pairs": candidates[:30],
        "high_window_count": high_count,
        "window_count": len(candidates),
        "reason": reason,
        "ee_method": "track_pair_box_sliding_window",
    }


def compute_kickboard_score(frames):
    """전동킥보드/스쿠터류 후보를 기존 world/two_wheeler feature에서 분리해 보조 event로 사용."""
    evidence = []
    hit_frames = 0
    rider_like_hits = 0
    total = max(1, len(frames))
    hints = ["kickboard", "electric scooter", "scooter", "moped"]

    for fr in frames:
        frame_hits = []
        for obj in (fr.get("two_wheelers", []) or []) + (fr.get("world_objects", []) or []):
            label = normalize_label(obj.get("class", ""))
            if any(h in label for h in hints):
                frame_hits.append(obj)
        if frame_hits:
            hit_frames += 1
            has_rider = bool(fr.get("rider_pairs"))
            if has_rider:
                rider_like_hits += 1
            best = sorted(frame_hits, key=lambda x: x.get("confidence", 0.0), reverse=True)[0]
            evidence.append({
                "frame_id": fr.get("frame_id"),
                "time_sec": fr.get("time_sec"),
                "class": best.get("class"),
                "confidence": best.get("confidence"),
                "bbox": best.get("bbox"),
                "has_rider_pair": has_rider,
            })

    hit_ratio = hit_frames / total
    rider_ratio = rider_like_hits / max(1, hit_frames)
    max_conf = max([_ee_float(e.get("confidence"), 0.0) for e in evidence], default=0.0)
    score = min(1.0, hit_ratio * 0.48 + rider_ratio * 0.22 + max_conf * 0.30)

    if score >= 0.38 and hit_frames >= 2:
        label = "electric_kickboard"
        reason = "전동킥보드/스쿠터류 객체가 여러 frame에서 확인됩니다."
    elif score >= 0.24 and hit_frames >= 1:
        label = "electric_kickboard_candidate"
        reason = "전동킥보드/스쿠터류 후보가 있으나 지속성 또는 rider 근거가 부족합니다."
    else:
        label = "not_electric_kickboard"
        reason = "전동킥보드 후보가 충분하지 않습니다."

    return {
        "kickboard_label": label,
        "electric_kickboard_score": round(float(score), 4),
        "is_electric_kickboard": label == "electric_kickboard",
        "hit_frames": hit_frames,
        "rider_like_hits": rider_like_hits,
        "hit_ratio": round(float(hit_ratio), 4),
        "rider_ratio": round(float(rider_ratio), 4),
        "best_evidence": evidence[:12],
        "reason": reason,
    }


def _ee_event_conf(score, label, strong_label, candidate_label=None, weight=1.0, cap=1.0):
    s = _ee_float(score, 0.0)
    if label == strong_label:
        conf = s * weight
    elif candidate_label and label == candidate_label:
        conf = s * min(weight, 0.92)
    else:
        conf = 0.0
    return round(float(min(cap, conf)), 4)


def fuse_events(fight, stalking, riding, helmet, intrusion_result=None, kickboard=None):
    """EE override: 후보는 모두 저장하되 final_label은 가장 강한 하나만 선택."""
    events = []

    # fight/stalking 충돌 시 fight의 접촉/급격한 움직임 근거를 우선한다.
    fight_conf = _ee_event_conf(fight.get("fight_score"), fight.get("fight_label"), "fight", "fight_candidate", weight=1.08)
    if fight.get("fight_label") in ["fight", "fight_candidate"]:
        events.append(normalize_event(
            "abnormal_action", "fight", fight.get("fight_label"), fight.get("fight_score", 0.0),
            confidence=fight_conf, evidence=fight.get("best_fight_window"), reason=fight.get("reason")
        ))

    stalk_conf = _ee_event_conf(stalking.get("stalking_score"), stalking.get("stalking_label"), "stalking", "stalking_candidate", weight=1.00)
    if fight_conf >= FINAL_MIN_EVENT_CONF:
        fw = fight.get("best_fight_window") or {}
        if _ee_float(fw.get("contact_ratio"), 0.0) >= 0.20 or _ee_float(fw.get("abrupt_score"), 0.0) >= 0.35:
            stalk_conf = round(float(stalk_conf * 0.72), 4)
    if stalking.get("stalking_label") in ["stalking", "stalking_candidate"]:
        events.append(normalize_event(
            "abnormal_action", "stalking", stalking.get("stalking_label"), stalking.get("stalking_score", 0.0),
            confidence=stalk_conf, evidence=stalking.get("best_pair"), reason=stalking.get("reason")
        ))

    if riding.get("riding_label") in ["sidewalk_riding", "sidewalk_candidate"]:
        subtype = "sidewalk_riding"
        label = riding.get("riding_label")
        conf = _ee_event_conf(riding.get("sidewalk_riding_score"), label, "sidewalk_riding", "sidewalk_candidate", weight=1.02)
        events.append(normalize_event(
            "traffic_violation", subtype, label, riding.get("sidewalk_riding_score", 0.0),
            confidence=conf, evidence=(riding.get("best_evidence") or [])[:5], reason=riding.get("reason")
        ))

    if helmet.get("helmet_label") == "no_helmet":
        conf = _ee_event_conf(helmet.get("no_helmet_score"), helmet.get("helmet_label"), "no_helmet", None, weight=0.98)
        events.append(normalize_event(
            "traffic_violation", "no_helmet", helmet.get("helmet_label"), helmet.get("no_helmet_score", 0.0),
            confidence=conf, evidence=(helmet.get("best_evidence") or [])[:5], reason=helmet.get("reason")
        ))

    if isinstance(kickboard, dict) and kickboard.get("kickboard_label") in ["electric_kickboard", "electric_kickboard_candidate"]:
        # 단순 객체 후보가 sidewalk_riding/fight/stalking을 덮지 않도록 confidence는 약간 낮춘다.
        label = kickboard.get("kickboard_label")
        conf = _ee_event_conf(kickboard.get("electric_kickboard_score"), label, "electric_kickboard", "electric_kickboard_candidate", weight=0.86)
        events.append(normalize_event(
            "traffic_object", "electric_kickboard", label, kickboard.get("electric_kickboard_score", 0.0),
            confidence=conf, evidence=(kickboard.get("best_evidence") or [])[:5], reason=kickboard.get("reason")
        ))

    if intrusion_result and intrusion_result.get("intrusion_label") in ["intrusion", "intrusion_candidate"]:
        for ev in intrusion_result.get("detected_events", []) or []:
            label = ev.get("label") or intrusion_result.get("intrusion_label")
            score = intrusion_result.get("intrusion_score", ev.get("score", ev.get("confidence", 0.0)))
            raw_conf = _ee_float(ev.get("confidence", score), 0.0)
            # 침입 후보가 너무 쉽게 final을 덮지 않도록 candidate는 낮추고, intrusion은 원래 근거를 반영한다.
            if label == "intrusion":
                conf = min(1.0, raw_conf * 0.96)
            else:
                conf = min(0.42, raw_conf * 0.82)
            events.append(normalize_event(
                "abnormal_path_intrusion", "intrusion", label, score,
                confidence=conf, evidence=ev.get("evidence"), reason=ev.get("reason")
            ))

    events = sorted(events, key=lambda x: (x.get("confidence", 0.0), x.get("score", 0.0)), reverse=True)
    primary = events[0] if events and events[0].get("confidence", 0.0) >= FINAL_MIN_EVENT_CONF else None

    final_label = primary.get("event_subtype") if primary else "other"

    return {
        "final_label": final_label,
        "final_primary_event": primary,
        "candidate_events": events,
        "detected_event_count": len([e for e in events if e.get("confidence", 0) >= FINAL_MIN_EVENT_CONF]),
        "one_label_only": True,
    }

print("EE enhanced reasoning override loaded")
print("candidates: fight, stalking, sidewalk_riding, no_helmet, electric_kickboard, intrusion, other")
print("changed part: fight/stalking feature scoring only; existing riding/helmet/kickboard/intrusion reasoning remains included")


EE enhanced reasoning override loaded
candidates: fight, stalking, sidewalk_riding, no_helmet, electric_kickboard, intrusion, other
changed part: fight/stalking feature scoring only; existing riding/helmet/kickboard/intrusion reasoning remains included


In [8]:

# ============================================================
# Cell 7. EE 최종 판정 실행
# ============================================================
feature_json_paths = sorted(CC_FEATURE_DIR.rglob("*.json"))
if not feature_json_paths and REUSE_CC_FEATURES_IF_DD_EMPTY and LEGACY_CC_FEATURE_DIR.exists():
    feature_json_paths = sorted(LEGACY_CC_FEATURE_DIR.rglob("*.json"))
    print("EE feature가 없어 기존 dd_features를 재사용합니다:", LEGACY_CC_FEATURE_DIR)
elif not feature_json_paths and REUSE_CC_FEATURES_IF_DD_EMPTY and 'LEGACY_CC_FEATURE_DIR_2' in globals() and LEGACY_CC_FEATURE_DIR_2.exists():
    feature_json_paths = sorted(LEGACY_CC_FEATURE_DIR_2.rglob("*.json"))
    print("EE/DD feature가 없어 기존 cc_features를 재사용합니다:", LEGACY_CC_FEATURE_DIR_2)
print("feature json 수:", len(feature_json_paths))
EE_EVENT_CANDIDATES = ["fight", "stalking", "sidewalk_riding", "no_helmet", "electric_kickboard", "intrusion", "other"]
print("EE final 후보:", EE_EVENT_CANDIDATES)
print("주의: candidate_events에는 여러 후보를 저장하지만 final_label은 아래 fusion에서 하나만 선택됩니다.")

results = []
failed_final = []

for jp in tqdm(feature_json_paths, desc="EE final reasoning"):
    try:
        with open(jp, "r", encoding="utf-8") as f:
            data = json.load(f)

        frames = data.get("frames", [])
        person_tracks = assign_tracks_from_frame_objects(frames, object_key="persons", fallback_max_dist=170, min_conf=PERSON_MIN_CONF)
        vehicle_tracks = assign_tracks_from_frame_objects(frames, object_key="two_wheelers", fallback_max_dist=220, min_conf=VEHICLE_MIN_CONF)

        fight = compute_fight_score(frames, person_tracks)
        stalking = compute_stalking_score(frames, person_tracks, fight_context=fight)
        riding = compute_sidewalk_riding_score(frames)
        helmet = compute_helmet_score(frames)
        kickboard = compute_kickboard_score(frames)
        intrusion = compute_integrated_intrusion_score(frames, person_tracks)

        fusion = fuse_events(
            fight=fight,
            stalking=stalking,
            riding=riding,
            helmet=helmet,
            intrusion_result=intrusion,
            kickboard=kickboard
        )

        track_summary = {
            str(tid): track_motion(tr)
            for tid, tr in person_tracks.items()
            if tr
        }
        vehicle_track_summary = {
            str(tid): track_motion(tr)
            for tid, tr in vehicle_tracks.items()
            if tr
        }

        item = {
            "video_name": data.get("video_name"),
            "video_path": data.get("video_path"),
            "relative_video_path": data.get("relative_video_path"),
            "final_label": fusion.get("final_label"),
            "final_primary_event": fusion.get("final_primary_event"),
            "candidate_events": fusion.get("candidate_events"),
            "action_label": (
                "fight" if fight.get("fight_label") == "fight"
                else "stalking" if stalking.get("stalking_label") == "stalking"
                else "other"
            ),
            "fight_label": fight.get("fight_label"),
            "stalking_label": stalking.get("stalking_label"),
            "riding_label": riding.get("riding_label"),
            "helmet_label": helmet.get("helmet_label"),
            "kickboard_label": kickboard.get("kickboard_label") if isinstance(kickboard, dict) else None,
            "intrusion_label": intrusion.get("intrusion_label") if isinstance(intrusion, dict) else None,
            "fight": {k: v for k, v in fight.items() if k != "windows"},
            "stalking": stalking,
            "riding": riding,
            "helmet": helmet,
            "kickboard": kickboard,
            "intrusion": intrusion,
            "track_summary": track_summary,
            "vehicle_track_summary": vehicle_track_summary,
            "model_status": data.get("settings", {}).get("model_status", MODEL_STATUS),
            "feature_json": str(jp),
        }

        results.append(item)

        # 개별 결과 저장
        stem = Path(jp).stem
        out_path = CC_FINAL_DIR / f"{stem}_ee_result.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(item, f, ensure_ascii=False, indent=2)

    except Exception as e:
        failed_final.append({"json": str(jp), "error": repr(e)})
        print("[final failed]", jp, repr(e))

summary = {
    "task": "ee_full_integrated_dd_base_plus_pair_window_fight_stalking",
    "cc_version": CC_VERSION,
    "video_root": str(VIDEO_ROOT),
    "output_dir": str(CC_DIR),
    "feature_dir": str(CC_FEATURE_DIR),
    "final_dir": str(CC_FINAL_DIR),
    "model_status": MODEL_STATUS,
    "settings": {
        "frame_interval": FRAME_INTERVAL,
        "fight_threshold": FIGHT_THRESHOLD,
        "stalking_threshold": STALKING_THRESHOLD,
        "sidewalk_riding_threshold": SIDEWALK_RIDING_THRESHOLD,
        "sidewalk_candidate_threshold": SIDEWALK_CANDIDATE_THRESHOLD,
        "no_helmet_threshold": NO_HELMET_THRESHOLD,
        "run_intrusion_integrated": RUN_INTRUSION_INTEGRATED,
        "intrusion_threshold": INTRUSION_THRESHOLD,
        "intrusion_candidate_threshold": INTRUSION_CANDIDATE_THRESHOLD,
        "ee_final_one_label_only": EE_FINAL_ONE_LABEL_ONLY,
    },
    "total_json": len(feature_json_paths),
    "success_count": len(results),
    "failed_count": len(failed_final),
    "final_label_count": dict(Counter(r.get("final_label") for r in results)),
    "action_label_count": dict(Counter(r.get("action_label") for r in results)),
    "fight_label_count": dict(Counter(r.get("fight_label") for r in results)),
    "stalking_label_count": dict(Counter(r.get("stalking_label") for r in results)),
    "riding_label_count": dict(Counter(r.get("riding_label") for r in results)),
    "helmet_label_count": dict(Counter(r.get("helmet_label") for r in results)),
    "kickboard_label_count": dict(Counter(r.get("kickboard_label") for r in results)),
    "intrusion_label_count": dict(Counter(r.get("intrusion_label") for r in results)),
    "results": results,
    "failed": failed_final,
}

summary_path = CC_FINAL_DIR / "ee_final_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("=" * 80)
print("EE 최종 판정 완료")
print("성공:", len(results), "실패:", len(failed_final))
print("summary 저장:", summary_path)
print("final_label_count:", summary["final_label_count"])
print("fight_label_count:", summary["fight_label_count"])
print("stalking_label_count:", summary["stalking_label_count"])
print("riding_label_count:", summary["riding_label_count"])
print("helmet_label_count:", summary["helmet_label_count"])
print("kickboard_label_count:", summary["kickboard_label_count"])
print("intrusion_label_count:", summary["intrusion_label_count"])
if failed_final:
    for x in failed_final[:10]:
        print(x)


feature json 수: 22
EE final 후보: ['fight', 'stalking', 'sidewalk_riding', 'no_helmet', 'electric_kickboard', 'intrusion', 'other']
주의: candidate_events에는 여러 후보를 저장하지만 final_label은 아래 fusion에서 하나만 선택됩니다.


EE final reasoning: 100%|██████████████████████████████████████████████████████████████| 22/22 [00:11<00:00,  1.90it/s]

EE 최종 판정 완료
성공: 22 실패: 0
summary 저장: C:\Users\우성한\outputs\prediction\ee\final_ee\ee_final_summary.json
final_label_count: {'intrusion': 12, 'fight': 2, 'electric_kickboard': 5, 'other': 1, 'sidewalk_riding': 2}
fight_label_count: {'fight_candidate': 2, 'not_fight': 16, 'fight': 4}
stalking_label_count: {'stalking': 2, 'not_stalking': 18, 'stalking_candidate': 2}
riding_label_count: {'road_riding_or_unknown': 3, 'sidewalk_riding': 12, 'sidewalk_candidate': 7}
helmet_label_count: {'helmet_unknown': 15, 'not_applicable': 3, 'helmet': 3, 'no_helmet': 1}
kickboard_label_count: {'electric_kickboard': 16, 'electric_kickboard_candidate': 3, 'not_electric_kickboard': 3}
intrusion_label_count: {'intrusion': 13, 'no_intrusion': 9}


In [9]:

# ============================================================
# Cell 8. EE 결과 보기 좋게 출력 + 텍스트 리포트 저장
# ============================================================
summary_path = CC_FINAL_DIR / "ee_final_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

results = summary.get("results", [])
print("total:", len(results))
print("final_label_count:", summary.get("final_label_count"))
print("fight_label_count:", summary.get("fight_label_count"))
print("stalking_label_count:", summary.get("stalking_label_count"))
print("riding_label_count:", summary.get("riding_label_count"))
print("helmet_label_count:", summary.get("helmet_label_count"))
print("kickboard_label_count:", summary.get("kickboard_label_count"))
print("intrusion_label_count:", summary.get("intrusion_label_count"))

def _score_of(r, path, default=0.0):
    cur = r
    for key in path:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key)
    return cur if isinstance(cur, (int, float)) else default

print("\n===== primary TOP 30 =====")
for r in sorted(results, key=lambda x: (x.get("final_primary_event") or {}).get("confidence", 0), reverse=True)[:30]:
    print({
        "video": r.get("video_name"),
        "final": r.get("final_label"),
        "primary": (r.get("final_primary_event") or {}).get("event_subtype"),
        "conf": (r.get("final_primary_event") or {}).get("confidence"),
        "fight": r.get("fight_label"),
        "stalking": r.get("stalking_label"),
        "riding": r.get("riding_label"),
        "helmet": r.get("helmet_label"),
        "kickboard": r.get("kickboard_label"),
    })

print("\n===== intrusion TOP 20 =====")
for r in sorted(results, key=lambda x: _score_of(x, ["intrusion", "intrusion_score"]), reverse=True)[:20]:
    print({
        "video": r.get("video_name"),
        "score": r.get("intrusion", {}).get("intrusion_score"),
        "label": r.get("intrusion_label"),
        "subtype": ((r.get("intrusion", {}).get("detected_events") or [{}])[0]).get("event_subtype"),
        "object": ((r.get("intrusion", {}).get("detected_events") or [{}])[0]).get("scene_object"),
        "reason": r.get("intrusion", {}).get("reason"),
    })

print("\n===== sidewalk TOP 20 =====")
for r in sorted(results, key=lambda x: _score_of(x, ["riding", "sidewalk_riding_score"]), reverse=True)[:20]:
    print({
        "video": r.get("video_name"),
        "score": r.get("riding", {}).get("sidewalk_riding_score"),
        "label": r.get("riding_label"),
        "tw_frames": r.get("riding", {}).get("tw_frame_count"),
        "rider_frames": r.get("riding", {}).get("rider_frame_count"),
        "best": r.get("riding", {}).get("best_evidence", [])[:1],
    })

print("\n===== helmet TOP 20 =====")
for r in sorted(results, key=lambda x: _score_of(x, ["helmet", "no_helmet_score"]), reverse=True)[:20]:
    print({
        "video": r.get("video_name"),
        "score": r.get("helmet", {}).get("no_helmet_score"),
        "label": r.get("helmet_label"),
        "rider_frames": r.get("helmet", {}).get("rider_frames"),
        "no_helmet_count": r.get("helmet", {}).get("no_helmet_count"),
        "candidate_count": r.get("helmet", {}).get("no_helmet_candidate_count"),
        "reason": r.get("helmet", {}).get("reason"),
    })

print("\n===== fight TOP 20 =====")
for r in sorted(results, key=lambda x: _score_of(x, ["fight", "fight_score"]), reverse=True)[:20]:
    print({
        "video": r.get("video_name"),
        "score": r.get("fight", {}).get("fight_score"),
        "label": r.get("fight_label"),
        "best_window": r.get("fight", {}).get("best_fight_window"),
    })

print("\n===== stalking TOP 20 =====")
for r in sorted(results, key=lambda x: _score_of(x, ["stalking", "stalking_score"]), reverse=True)[:20]:
    print({
        "video": r.get("video_name"),
        "score": r.get("stalking", {}).get("stalking_score"),
        "label": r.get("stalking_label"),
        "best_pair": r.get("stalking", {}).get("best_pair"),
    })

report_lines = []
report_lines.append("CC FINAL REPORT")
report_lines.append(f"summary: {summary_path}")
report_lines.append(f"total: {len(results)}")
report_lines.append(f"final_label_count: {summary.get('final_label_count')}")
report_lines.append(f"fight_label_count: {summary.get('fight_label_count')}")
report_lines.append(f"stalking_label_count: {summary.get('stalking_label_count')}")
report_lines.append(f"riding_label_count: {summary.get('riding_label_count')}")
report_lines.append(f"helmet_label_count: {summary.get('helmet_label_count')}")
report_lines.append("")
for r in sorted(results, key=lambda x: (x.get("final_primary_event") or {}).get("confidence", 0), reverse=True):
    p = r.get("final_primary_event") or {}
    report_lines.append(
        f"{r.get('video_name')} | final={r.get('final_label')} | "
        f"primary={p.get('event_subtype')} conf={p.get('confidence')} | "
        f"fight={r.get('fight_label')}:{r.get('fight', {}).get('fight_score')} | "
        f"stalking={r.get('stalking_label')}:{r.get('stalking', {}).get('stalking_score')} | "
        f"riding={r.get('riding_label')}:{r.get('riding', {}).get('sidewalk_riding_score')} | "
        f"helmet={r.get('helmet_label')}:{r.get('helmet', {}).get('no_helmet_score')}"
    )

report_path = CC_FINAL_DIR / "cc_final_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print("\n리포트 저장:", report_path)


total: 22
final_label_count: {'intrusion': 12, 'fight': 2, 'electric_kickboard': 5, 'other': 1, 'sidewalk_riding': 2}
fight_label_count: {'fight_candidate': 2, 'not_fight': 16, 'fight': 4}
stalking_label_count: {'stalking': 2, 'not_stalking': 18, 'stalking_candidate': 2}
riding_label_count: {'road_riding_or_unknown': 3, 'sidewalk_riding': 12, 'sidewalk_candidate': 7}
helmet_label_count: {'helmet_unknown': 15, 'not_applicable': 3, 'helmet': 3, 'no_helmet': 1}
kickboard_label_count: {'electric_kickboard': 16, 'electric_kickboard_candidate': 3, 'not_electric_kickboard': 3}
intrusion_label_count: {'intrusion': 13, 'no_intrusion': 9}

===== primary TOP 30 =====
{'video': 'md_e2122_c2.mp4', 'final': 'fight', 'primary': 'fight', 'conf': 0.8945, 'fight': 'fight', 'stalking': 'stalking_candidate', 'riding': 'sidewalk_candidate', 'helmet': 'no_helmet', 'kickboard': 'electric_kickboard'}
{'video': 'lo_e1281_c2.mp4', 'final': 'intrusion', 'primary': 'intrusion', 'conf': 0.8928, 'fight': 'not_fight

In [10]:

# ============================================================
# Cell 9. EE 자연어 비슷한 검색 함수
# ============================================================
def search_cc_results(query, top_k=50):
    q = str(query).lower()
    wants_fight = any(k in q for k in ["싸움", "폭행", "fight", "violence", "충돌"])
    wants_stalking = any(k in q for k in ["스토킹", "따라", "추적", "stalking", "follow"])
    wants_sidewalk = any(k in q for k in ["인도", "보도", "sidewalk", "침범"])
    wants_tw = any(k in q for k in ["오토바이", "자전거", "킥보드", "스쿠터", "이륜", "motorcycle", "bicycle", "scooter", "kickboard"])
    wants_kickboard = any(k in q for k in ["킥보드", "전동킥보드", "electric scooter", "kickboard", "scooter"])
    wants_no_helmet = any(k in q for k in ["헬멧", "노헬멧", "미착용", "no helmet", "no_helmet", "without helmet"])
    wants_intrusion = any(k in q for k in ["침입", "경계", "울타리", "게이트", "intrusion", "gate", "fence"])

    hits = []
    for r in results:
        ok = False
        score = 0.0

        if wants_fight:
            s = r.get("fight", {}).get("fight_score", 0.0)
            if r.get("fight_label") in ["fight", "fight_candidate"]:
                ok = True
                score += s

        if wants_stalking:
            s = r.get("stalking", {}).get("stalking_score", 0.0)
            if r.get("stalking_label") in ["stalking", "stalking_candidate"]:
                ok = True
                score += s

        if wants_sidewalk or wants_tw:
            s = r.get("riding", {}).get("sidewalk_riding_score", 0.0)
            if r.get("riding_label") in ["sidewalk_riding", "sidewalk_candidate"]:
                ok = True
                score += s

        if wants_kickboard:
            s = r.get("kickboard", {}).get("electric_kickboard_score", 0.0)
            if r.get("kickboard_label") in ["electric_kickboard", "electric_kickboard_candidate"]:
                ok = True
                score += s

        if wants_no_helmet:
            s = r.get("helmet", {}).get("no_helmet_score", 0.0)
            if r.get("helmet_label") in ["no_helmet", "helmet_unknown"]:
                ok = True
                score += s if r.get("helmet_label") == "no_helmet" else 0.05

        if wants_intrusion:
            for ev in r.get("candidate_events", []) or []:
                if ev.get("event_type") == "abnormal_path_intrusion":
                    ok = True
                    score += ev.get("confidence", 0.0)
            intr = r.get("intrusion", {})
            if intr.get("intrusion_label") in ["intrusion", "intrusion_candidate"]:
                ok = True
                score += intr.get("intrusion_score", 0.0)

        if not any([wants_fight, wants_stalking, wants_sidewalk, wants_tw, wants_no_helmet, wants_intrusion]):
            # 키워드가 없으면 final_label/video_name 검색
            text = json.dumps(r, ensure_ascii=False).lower()
            if q in text:
                ok = True
                score += (r.get("final_primary_event") or {}).get("confidence", 0.0)

        if ok:
            hits.append((score, r))

    hits = sorted(hits, key=lambda x: x[0], reverse=True)[:top_k]
    for score, r in hits:
        print({
            "score": round(float(score), 4),
            "video": r.get("video_name"),
            "final": r.get("final_label"),
            "primary": r.get("final_primary_event"),
            "fight": (r.get("fight_label"), r.get("fight", {}).get("fight_score")),
            "stalking": (r.get("stalking_label"), r.get("stalking", {}).get("stalking_score")),
            "riding": (r.get("riding_label"), r.get("riding", {}).get("sidewalk_riding_score")),
            "helmet": (r.get("helmet_label"), r.get("helmet", {}).get("no_helmet_score")),
        })
    return [r for _, r in hits]


# 사용 예시
# search_cc_results("인도 주행 이륜차")
# search_cc_results("헬멧 미착용")
# search_cc_results("싸움")
# search_cc_results("스토킹")


In [11]:

# ============================================================
# Cell X. EE 결과 한눈에 보기 Dashboard
#   - ee_final_summary.json을 표 형태로 요약
#   - final_label은 영상당 하나만 표시
#   - fight/stalking/riding/helmet/kickboard/intrusion 점수를 한 행에 표시
#   - CSV 리포트 저장
# ============================================================
from pathlib import Path
import json
from collections import Counter

try:
    import pandas as pd
except Exception as e:
    pd = None
    print("pandas import 실패. 기본 print 모드로 출력합니다:", repr(e))

try:
    from IPython.display import display
except Exception:
    display = print


def _safe_get(d, keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k)
    return default if cur is None else cur


def _round_score(x):
    try:
        return round(float(x), 4)
    except Exception:
        return x


def _short_text(x, n=100):
    if x is None:
        return ""
    s = str(x).replace("\n", " ")
    return s if len(s) <= n else s[:n] + "..."


def _best_evidence_text(r):
    primary = r.get("final_primary_event") or {}
    subtype = primary.get("event_subtype") or primary.get("event_type") or r.get("final_label") or ""

    if subtype == "fight" or r.get("fight_label") in ["fight", "fight_candidate"]:
        ev = _safe_get(r, ["fight", "best_fight_window"], {}) or {}
        return f"fight {ev.get('start_sec')}~{ev.get('end_sec')} pair={ev.get('pair')} contact={ev.get('contact_ratio')} abrupt={ev.get('abrupt_score')}"

    if subtype == "stalking" or r.get("stalking_label") in ["stalking", "stalking_candidate"]:
        ev = _safe_get(r, ["stalking", "best_pair"], {}) or {}
        return f"stalking {ev.get('start_sec')}~{ev.get('end_sec')} pair={ev.get('pair')} dist={ev.get('distance_mean')} same_dir={ev.get('same_direction_ratio')}"

    if subtype == "sidewalk_riding" or r.get("riding_label") in ["sidewalk_riding", "sidewalk_candidate"]:
        evs = _safe_get(r, ["riding", "best_evidence"], []) or []
        ev = evs[0] if evs else {}
        return f"riding frame={ev.get('frame_id')} sidewalk={ev.get('score') or ev.get('sidewalk_score')} rider={ev.get('best_rider_pair_score')}"

    if subtype == "no_helmet" or r.get("helmet_label") == "no_helmet":
        return _short_text(_safe_get(r, ["helmet", "reason"], ""), 100)

    if subtype == "electric_kickboard" or r.get("kickboard_label") in ["electric_kickboard", "electric_kickboard_candidate"]:
        evs = _safe_get(r, ["kickboard", "best_evidence"], []) or []
        ev = evs[0] if evs else {}
        return f"kickboard frame={ev.get('frame_id')} class={ev.get('class')} conf={ev.get('confidence')}"

    if subtype == "intrusion" or "intrusion" in str(subtype):
        return _short_text(primary.get("reason") or _safe_get(r, ["intrusion", "reason"], ""), 100)

    return _short_text(primary.get("reason") or r.get("final_label"), 100)


def load_ee_summary(summary_path=None):
    if summary_path is None:
        if "CC_FINAL_DIR" in globals():
            summary_path = Path(CC_FINAL_DIR) / "ee_final_summary.json"
        else:
            summary_path = Path(r"C:\Users\우성한\outputs\prediction\ee\final_ee\ee_final_summary.json")

    summary_path = Path(summary_path)
    if not summary_path.exists():
        raise FileNotFoundError(
            f"summary 파일을 찾지 못했습니다: {summary_path}\n"
            "먼저 Cell 7 최종 판정을 실행해서 ee_final_summary.json을 생성하세요."
        )

    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)
    return summary_path, summary


def make_ee_overview(summary_path=None, top_n=100, save_csv=True, only_detected=False):
    summary_path, summary = load_ee_summary(summary_path)
    results = summary.get("results", []) or []

    rows = []
    for r in results:
        primary = r.get("final_primary_event") or {}
        candidates = r.get("candidate_events") or []
        row = {
            "video": r.get("video_name"),
            "final": r.get("final_label"),
            "primary": primary.get("event_subtype") or primary.get("event_type"),
            "primary_conf": _round_score(primary.get("confidence", 0.0)),
            "candidate_count": len(candidates),
            "fight": r.get("fight_label"),
            "fight_score": _round_score(_safe_get(r, ["fight", "fight_score"], 0.0)),
            "stalking": r.get("stalking_label"),
            "stalking_score": _round_score(_safe_get(r, ["stalking", "stalking_score"], 0.0)),
            "riding": r.get("riding_label"),
            "riding_score": _round_score(_safe_get(r, ["riding", "sidewalk_riding_score"], 0.0)),
            "helmet": r.get("helmet_label"),
            "helmet_score": _round_score(_safe_get(r, ["helmet", "no_helmet_score"], 0.0)),
            "kickboard": r.get("kickboard_label"),
            "kickboard_score": _round_score(_safe_get(r, ["kickboard", "electric_kickboard_score"], 0.0)),
            "intrusion": r.get("intrusion_label"),
            "intrusion_score": _round_score(_safe_get(r, ["intrusion", "intrusion_score"], 0.0)),
            "evidence": _best_evidence_text(r),
        }
        if only_detected and row["final"] == "other":
            continue
        rows.append(row)

    rows = sorted(rows, key=lambda x: x.get("primary_conf", 0.0) or 0.0, reverse=True)

    print("=" * 90)
    print("EE 전체 통합 결과 요약 - final_label은 영상당 하나")
    print("summary:", summary_path)
    print("total:", len(results), "display:", min(top_n, len(rows)))
    print("success:", summary.get("success_count"), "failed:", summary.get("failed_count"))
    print("final_label_count:", dict(Counter([r.get("final") for r in rows])))
    print("=" * 90)

    if pd is not None:
        df = pd.DataFrame(rows)
        display(df.head(top_n))
        if save_csv:
            out_csv = summary_path.parent / "ee_overview_table.csv"
            df.to_csv(out_csv, index=False, encoding="utf-8-sig")
            print("CSV 저장:", out_csv)
        return df

    for row in rows[:top_n]:
        print(row)
    return rows


overview_rows = make_ee_overview(top_n=100, save_csv=True, only_detected=False)


EE 전체 통합 결과 요약 - final_label은 영상당 하나
summary: C:\Users\우성한\outputs\prediction\ee\final_ee\ee_final_summary.json
total: 22 display: 22
success: 22 failed: 0
final_label_count: {'fight': 2, 'intrusion': 12, 'electric_kickboard': 5, 'sidewalk_riding': 2, 'other': 1}


,video,final,primary,primary_conf,candidate_count,fight,fight_score,stalking,stalking_score,riding,riding_score,helmet,helmet_score,kickboard,kickboard_score,intrusion,intrusion_score,evidence
0,md_e2122_c2.mp4,fight,fight,0.8945,6,fight,0.8282,stalking_candidate,0.3602,sidewalk_candidate,0.3014,no_helmet,0.3659,electric_kickboard,0.6055,intrusion,0.7683,"fight 30.0~34.0 pair=[36, 70] contact=1.0 abru..."
1,lo_e1281_c2.mp4,intrusion,intrusion,0.8928,3,not_fight,0.0212,not_stalking,0.0000,sidewalk_riding,0.4868,helmet_unknown,0.3028,electric_kickboard,0.6978,intrusion,0.7740,riding frame=88 sidewalk=0.385 rider=1.0
2,fp_e1520_c1.mp4,intrusion,intrusion,0.8928,3,not_fight,0.0370,not_stalking,0.0000,sidewalk_candidate,0.2685,helmet_unknown,0.0155,electric_kickboard_candidate,0.3503,intrusion,0.8000,riding frame=384 sidewalk=0.3503 rider=0.5724
3,ph_e2063_c1.mp4,intrusion,intrusion,0.8928,5,fight,0.8164,stalking,0.5163,sidewalk_candidate,0.3488,helmet_unknown,0.0662,electric_kickboard_candidate,0.3580,intrusion,0.8037,"fight 15.0~19.0 pair=[3, 6] contact=0.6667 abr..."
4,es_e3768_c1.mp4,intrusion,intrusion,0.8928,4,fight_candidate,0.4159,not_stalking,0.0000,sidewalk_riding,0.5335,helmet_unknown,0.0800,electric_kickboard,0.8219,intrusion,0.8001,"fight 10.0~14.0 pair=[12, 13] contact=0.0 abru..."
5,es_e3768_c2.mp4,intrusion,intrusion,0.8928,3,not_fight,0.0841,not_stalking,0.0000,sidewalk_riding,0.5416,helmet_unknown,0.0800,electric_kickboard,0.8386,intrusion,0.8446,riding frame=896 sidewalk=0.6407 rider=0.9261
6,ab_e0457_c1.mp4,intrusion,intrusion,0.8928,3,not_fight,0.0402,not_stalking,0.0000,sidewalk_riding,0.5537,helmet_unknown,0.1976,electric_kickboard,0.4821,intrusion,0.8699,riding frame=392 sidewalk=0.63 rider=0.5642
7,mp_e2730_c1.mp4,intrusion,intrusion,0.8851,3,not_fight,0.0400,not_stalking,0.0000,sidewalk_candidate,0.2842,helmet,0.3418,electric_kickboard,0.5485,intrusion,0.7275,riding frame=232 sidewalk=0.385 rider=1.0
8,ph_e2063_c2.mp4,fight,fight,0.8834,4,fight,0.8180,stalking_candidate,0.4487,sidewalk_riding,0.5193,helmet_unknown,0.0800,electric_kickboard,0.4037,no_intrusion,0.0000,"fight 23.0~27.0 pair=[21, 22] contact=1.0 abru..."
9,mp_e2730_c2.mp4,intrusion,intrusion,0.8746,3,not_fight,0.0594,not_stalking,0.0000,sidewalk_candidate,0.2734,helmet,0.3406,electric_kickboard,0.6454,intrusion,0.7087,riding frame=304 sidewalk=0.385 rider=1.0


CSV 저장: C:\Users\우성한\outputs\prediction\ee\final_ee\ee_overview_table.csv


In [1]:
# ------------------------------------------------------------
# 5. 현재 메모리에서 예측 결과 자동 탐색
# ------------------------------------------------------------

prediction_records = None
prediction_source = None

preferred_names = [
    "batch_results",
    "all_results",
    "results",
    "processed_results",
    "final_results",
    "integrated_results",
    "video_results",
    "predictions",
    "summary",
    "batch_summary",
]

# 1차: 일반적으로 사용하는 변수명 검색
for variable_name in preferred_names:
    if variable_name not in globals():
        continue

    candidate_obj = globals()[variable_name]
    found_records = find_prediction_records(candidate_obj)

    if found_records:
        prediction_records = found_records
        prediction_source = f"memory variable: {variable_name}"
        break


# 2차: 메모리의 모든 list, dict 변수 탐색
if prediction_records is None:
    for variable_name, candidate_obj in list(globals().items()):
        if variable_name.startswith("_"):
            continue

        if not isinstance(candidate_obj, (list, dict)):
            continue

        try:
            found_records = find_prediction_records(
                candidate_obj
            )
        except Exception:
            continue

        valid_video_records = []

        for record in found_records:
            if get_record_video_name(record) is not None:
                valid_video_records.append(record)

        if valid_video_records:
            prediction_records = valid_video_records
            prediction_source = (
                f"auto-detected memory variable: "
                f"{variable_name}"
            )
            break


# 3차: 그래도 없으면 JSON 자동 검색
if prediction_records is None:
    search_roots = [
        Path(r"C:\Users\우성한\outputs\prediction"),
        Path.cwd(),
    ]

    checked_files = []

    for search_root in search_roots:
        if not search_root.exists():
            continue

        for json_path in search_root.rglob("*.json"):
            checked_files.append(str(json_path))

            try:
                with open(
                    json_path,
                    "r",
                    encoding="utf-8"
                ) as f:
                    candidate_json = json.load(f)

                found_records = find_prediction_records(
                    candidate_json
                )

                valid_video_records = []

                for record in found_records:
                    if get_record_video_name(record) is not None:
                        valid_video_records.append(record)

                if valid_video_records:
                    prediction_records = valid_video_records
                    prediction_source = str(json_path)
                    break

            except Exception:
                continue

        if prediction_records is not None:
            break


if prediction_records is None:
    raise RuntimeError(
        "영상별 예측 결과를 찾지 못했습니다.\n"
        "통합 추론 셀을 먼저 실행하거나, 추론 결과를 담은 "
        "변수명을 평가 코드에 직접 지정해야 합니다."
    )


print("prediction source:", prediction_source)
print("prediction record count:", len(prediction_records))

first_record = prediction_records[0]

print()
print("[첫 번째 결과 구조]")
print("keys:", first_record.keys())
print(first_record)

NameError: name 'Path' is not defined